# air-gap DB clone — profiler + generator

Инструмент «безопасного клонирования» БД через воздушный зазор из **двух отдельных программ**. Через зазор едет **не выгрузка данных, а маленький человекочитаемый `profile.json`** (схема + статистика) — его можно глазами проверить перед переносом.

```
закрытый контур (реальная БД)                 открытый контур (LLM, без реальной БД)
┌───────────────────────────┐   profile.json  ┌────────────────────────────────────┐
│  profiler                 │  ─────────────▶ │  generator                         │
│  каталог + pg_stats        │   (+ policy)    │  schema.sql + синтетические данные  │
│  → profile.json (аудит)    │                 │  (CSV или INSERT-ы)                │
└───────────────────────────┘                 └────────────────────────────────────┘
```

Этот ноутбук:
1. пересоздаёт весь инструмент в папке **`./generated`** (ячейки `%%writefile`);
2. запускает **profiler** на подключении БД проекта (`config_db.json`/`.env`); если БД недоступна — берёт готовый пример профиля;
3. запускает **generator** → `schema.sql` + синтетические данные в **`./generated/output`**.

> Все файлы генерируются в папке `generated/` рядом с этим ноутбуком.

## 1. Входные данные и настройки

Список таблиц задаётся **через запятую** в формате `schema.table` (или CSV-файлом).

In [ ]:
# ── Входные данные ─────────────────────────────────────────────────────────
# Таблицы для обработки: schema.table через запятую.
TABLES = "public.clients,public.tasks,public.payments"
# Либо путь к CSV со списком таблиц (колонки schema,table). Если задан — приоритетнее.
TABLES_CSV = ""            # напр. "generated/tables.example.csv"

# ── Настройки ──────────────────────────────────────────────────────────────
POLICY_YAML   = "generated/policy.example.yaml"   # whitelist чувствительности (или "")
SCALE_FACTOR  = 0.001      # доля от исходного числа строк (1.2M × 0.001 ≈ 1200)
SEED          = 42         # детерминизм генератора
OUTPUT_FORMAT = "csv"      # "csv" (файл на таблицу) | "sql" (батч INSERT-ов)
SAMPLE        = True       # инспектировать форматы коротких текстовых полей
KEEP_GPDB     = False      # сохранять DISTRIBUTED BY / партиции в DDL

from pathlib import Path
NB_DIR = Path.cwd()                 # папка с этим ноутбуком (air_gap_clone/)
GEN = NB_DIR / "generated"          # сюда генерируются ВСЕ файлы инструмента
OUT = GEN / "output"                # артефакты запуска (profile.json, schema.sql, данные)
OUT.mkdir(parents=True, exist_ok=True)
print("Файлы инструмента →", GEN)
print("Артефакты запуска →", OUT)

## 2. Генерация файлов инструмента в `./generated`

Ячейки ниже (`%%writefile`) записывают модули обеих программ на диск. Это и есть готовый к переносу инструмент — его можно запускать и отдельно от ноутбука (`python generated/profiler.py ...`, `python generated/generator.py ...`).

In [ ]:
# Каталоги инструмента.
for d in ["generated", "generated/agc_profiler", "generated/agc_generator", "generated/examples"]:
    Path(d).mkdir(parents=True, exist_ok=True)
print("Каталоги готовы")

### Общий модуль

`agc_common.py` — валидация SQL-идентификаторов и логирование, используется обеими программами.

In [ ]:
%%writefile generated/agc_common.py
"""Общие утилиты обеих программ: валидация идентификаторов, логирование.

Никакой бизнес-логики — только то, что нужно и профайлеру, и генератору.
"""
from __future__ import annotations

import logging
import re
import sys

# SQL-идентификатор: буква/underscore, далее буквы/цифры/underscore/$.
# Всё, что не проходит, — не подставляем в SQL как имя объекта (защита от инъекции).
_IDENT_RE = re.compile(r"^[A-Za-z_][A-Za-z0-9_$]*$")


def validate_identifier(name: str, kind: str = "identifier") -> str:
    """Проверяет, что name — безопасный SQL-идентификатор. Иначе ValueError.

    Значения в запросы всегда идут параметрами; идентификаторы (schema/table/
    column) параметризовать нельзя, поэтому валидируем их этой функцией.
    """
    if not isinstance(name, str) or not _IDENT_RE.match(name):
        raise ValueError(f"Недопустимый SQL-{kind}: {name!r}")
    return name


def quote_ident(name: str) -> str:
    """Экранирует идентификатор двойными кавычками (для DDL/SQL)."""
    return '"' + name.replace('"', '""') + '"'


def qualified(schema: str, table: str) -> str:
    return f"{quote_ident(schema)}.{quote_ident(table)}"


def get_logger(name: str = "agc") -> logging.Logger:
    """Единый логгер в stderr. Пишем: что прочитано, что засинтезировано, warning-и."""
    logger = logging.getLogger(name)
    if not logger.handlers:
        handler = logging.StreamHandler(sys.stderr)
        handler.setFormatter(
            logging.Formatter("%(asctime)s %(levelname)-7s [%(name)s] %(message)s", "%H:%M:%S")
        )
        logger.addHandler(handler)
        logger.setLevel(logging.INFO)
    return logger


### Программа 1 — profiler (закрытый контур)

Модули: `db` (подключение, read-only) · `catalog_reader` (структура из каталога) · `stats_reader` (pg_stats) · `sampler` (адаптивный сэмпл форматов) · `classifier` (эвристики-предложения) · `policy` (whitelist) · `linter` (аудит утечек) · `profile_builder` (сборка) · `cli`.

In [ ]:
%%writefile generated/agc_profiler/__init__.py
"""Программа 1 — profiler (закрытый контур).

Читает схему и статистику реальной БД из системного каталога и pg_stats,
применяет whitelist-политику чувствительности и пишет компактный profile.json.
Реальные строки данных наружу не выгружаются (см. linter — граница аудита).
"""


In [ ]:
%%writefile generated/agc_profiler/db.py
"""Подключение к БД закрытого контура (единственный сетевой вызов инструмента).

Порядок разрешения DSN:
  1) явный аргумент dsn / --dsn;
  2) переменные окружения AGC_DB_DSN, затем DB_DSN;
  3) db_config.json рядом с инструментом (ключ "dsn" или user_id/host/port/database);
  4) подключение проекта text2sql (src/text2sql/db/connection.py), если пакет
     доступен на PYTHONPATH — так инструмент переиспользует уже настроенный
     коннект проекта (config_db.json / .env), как просили.

Движок — read-only: default_transaction_read_only=on + statement_timeout,
чтобы профайлер физически не мог ничего записать в реальную БД.
"""
from __future__ import annotations

import json
import os
from pathlib import Path

from sqlalchemy import create_engine
from sqlalchemy.engine import Engine

from agc_common import get_logger

log = get_logger("profiler.db")


def _from_project() -> str | None:
    """DSN из проекта text2sql, если он импортируется (запуск внутри репозитория)."""
    try:
        from text2sql.config import DB  # type: ignore
        from text2sql.db.connection import load_connection  # type: ignore

        conn = load_connection()
        if conn and conn.is_complete():
            return conn.url()
        if DB.dsn:
            return DB.dsn
    except Exception:  # noqa: BLE001 — проект недоступен, это не ошибка
        return None
    return None


def resolve_dsn(dsn: str | None = None, config_path: str | Path | None = None) -> str:
    if dsn:
        return dsn
    for env in ("AGC_DB_DSN", "DB_DSN"):
        if os.getenv(env):
            return os.environ[env]
    if config_path and Path(config_path).exists():
        data = json.loads(Path(config_path).read_text(encoding="utf-8"))
        if data.get("dsn"):
            return str(data["dsn"])
        user = data.get("user_id") or data.get("user") or ""
        host = data["host"]
        port = data.get("port", 5432)
        database = data["database"]
        return f"postgresql://{user}@{host}:{port}/{database}"
    proj = _from_project()
    if proj:
        return proj
    raise RuntimeError(
        "Не задан DSN БД. Укажите --dsn, переменную AGC_DB_DSN/DB_DSN, "
        "db_config.json или запустите внутри проекта text2sql (config_db.json/.env)."
    )


def make_engine(
    dsn: str | None = None,
    config_path: str | Path | None = None,
    statement_timeout_ms: int = 600_000,
) -> Engine:
    """Read-only SQLAlchemy-движок к Greenplum/Postgres (драйвер psycopg2)."""
    url = resolve_dsn(dsn, config_path)
    # Прячем всё до '@' в логе (host/db показываем, user/креды — нет).
    safe = url.split("@", 1)[-1] if "@" in url else url
    log.info("Подключение к БД (read-only): ...@%s", safe)
    opts = (
        f"-c statement_timeout={int(statement_timeout_ms)} "
        f"-c default_transaction_read_only=on"
    )
    return create_engine(url, pool_pre_ping=True, connect_args={"options": opts})


In [ ]:
%%writefile generated/agc_profiler/catalog_reader.py
"""Чтение СТРУКТУРЫ из системного каталога — это ground truth, дёшево и без скана.

Читаем: типы/nullability/default, PK/UNIQUE/NOT NULL/FK/CHECK, ключ распределения
Greenplum (gp_distribution_policy), партиционирование, тип хранения (heap /
append-optimized / column-oriented), relkind (таблица/вью).

Версия GPDB заранее неизвестна — GPDB-специфику (gp_distribution_policy,
pg_partitions, pg_appendonly) оборачиваем в try/except и мягко деградируем.
"""
from __future__ import annotations

import re

from sqlalchemy import text
from sqlalchemy.engine import Engine

from agc_common import get_logger, validate_identifier

log = get_logger("profiler.catalog")

_NUMTYPE_RE = re.compile(r"^(?:numeric|decimal)\s*\(\s*(\d+)\s*,\s*(\d+)\s*\)", re.IGNORECASE)
_CHARLEN_RE = re.compile(r"\(\s*(\d+)\s*\)")

_RELKIND = {
    "r": "table", "v": "view", "m": "matview",
    "f": "foreign_table", "p": "partitioned_table", "t": "toast",
}


def read_table_meta(engine: Engine, schema: str, table: str) -> dict:
    """relkind, оценка reltuples, oid, тип хранения."""
    validate_identifier(schema, "schema")
    validate_identifier(table, "table")
    sql = text(
        "SELECT c.oid AS oid, c.relkind AS relkind, c.reltuples::bigint AS reltuples "
        "FROM pg_class c JOIN pg_namespace n ON n.oid = c.relnamespace "
        "WHERE n.nspname = :s AND c.relname = :t"
    )
    with engine.connect() as conn:
        row = conn.execute(sql, {"s": schema, "t": table}).mappings().first()
    if row is None:
        raise LookupError(f"Объект {schema}.{table} не найден в pg_class")
    oid = int(row["oid"])
    reltuples = int(row["reltuples"] or 0)
    meta = {
        "relkind": _RELKIND.get(row["relkind"], row["relkind"]),
        "is_view": row["relkind"] in ("v", "m"),
        "reltuples": reltuples,
        # reltuples всегда лишь оценка (обновляется ANALYZE); 0 — почти наверняка stale.
        "row_count_estimated": True,
        "storage": _read_storage(engine, oid),
        "oid": oid,
    }
    return meta


def _read_storage(engine: Engine, oid: int) -> str:
    """heap / append_optimized_row / append_optimized_column. GPDB-специфика — guarded."""
    try:
        with engine.connect() as conn:
            row = conn.execute(
                text("SELECT columnstore FROM pg_appendonly WHERE relid = :oid"),
                {"oid": oid},
            ).mappings().first()
        if row is None:
            return "heap"
        return "append_optimized_column" if row["columnstore"] else "append_optimized_row"
    except Exception:  # noqa: BLE001 — не GPDB / нет pg_appendonly
        return "heap"


def read_columns(engine: Engine, schema: str, table: str) -> list[dict]:
    """Колонки: имя, точный pg_type (format_type), nullability, default, precision/scale."""
    validate_identifier(schema, "schema")
    validate_identifier(table, "table")
    sql = text(
        "SELECT a.attnum AS attnum, a.attname AS name, "
        "       format_type(a.atttypid, a.atttypmod) AS pg_type, "
        "       a.attnotnull AS notnull, "
        "       pg_get_expr(ad.adbin, ad.adrelid) AS default_expr "
        "FROM pg_attribute a "
        "JOIN pg_class c ON c.oid = a.attrelid "
        "JOIN pg_namespace n ON n.oid = c.relnamespace "
        "LEFT JOIN pg_attrdef ad ON ad.adrelid = a.attrelid AND ad.adnum = a.attnum "
        "WHERE n.nspname = :s AND c.relname = :t AND a.attnum > 0 AND NOT a.attisdropped "
        "ORDER BY a.attnum"
    )
    cols: list[dict] = []
    with engine.connect() as conn:
        for row in conn.execute(sql, {"s": schema, "t": table}).mappings():
            pg_type = str(row["pg_type"])
            precision, scale = _parse_numeric(pg_type)
            cols.append({
                "name": str(row["name"]),
                "pg_type": pg_type,
                "nullable": not bool(row["notnull"]),
                "default": row["default_expr"],
                "precision": precision,
                "scale": scale,
                "char_len": _parse_charlen(pg_type),
                "ordinal": int(row["attnum"]),
            })
    return cols


def _parse_numeric(pg_type: str) -> tuple[int | None, int | None]:
    m = _NUMTYPE_RE.match(pg_type)
    return (int(m.group(1)), int(m.group(2))) if m else (None, None)


def _parse_charlen(pg_type: str) -> int | None:
    if pg_type.lower().startswith(("character", "varchar", "char")):
        m = _CHARLEN_RE.search(pg_type)
        return int(m.group(1)) if m else None
    return None


def read_constraints(engine: Engine, schema: str, table: str) -> dict:
    """PK/UNIQUE/FK/CHECK. Точный путь через pg_constraint (unnest WITH ORDINALITY);
    при недоступности (старая GPDB) — fallback на information_schema."""
    validate_identifier(schema, "schema")
    validate_identifier(table, "table")
    try:
        return _read_constraints_pg(engine, schema, table)
    except Exception as exc:  # noqa: BLE001
        log.warning("pg_constraint недоступен для %s.%s (%s) — fallback information_schema",
                    schema, table, exc)
        return _read_constraints_infoschema(engine, schema, table)


def _read_constraints_pg(engine: Engine, schema: str, table: str) -> dict:
    sql = text(
        "SELECT con.contype AS contype, con.conname AS conname, "
        "       pg_get_constraintdef(con.oid) AS def, "
        "       ARRAY(SELECT att.attname FROM unnest(con.conkey) WITH ORDINALITY k(a, o) "
        "             JOIN pg_attribute att ON att.attrelid = con.conrelid AND att.attnum = k.a "
        "             ORDER BY k.o) AS cols, "
        "       n2.nspname AS ref_schema, cl2.relname AS ref_table, "
        "       ARRAY(SELECT att.attname FROM unnest(con.confkey) WITH ORDINALITY k(a, o) "
        "             JOIN pg_attribute att ON att.attrelid = con.confrelid AND att.attnum = k.a "
        "             ORDER BY k.o) AS ref_cols "
        "FROM pg_constraint con "
        "JOIN pg_class cl ON cl.oid = con.conrelid "
        "JOIN pg_namespace n ON n.oid = cl.relnamespace "
        "LEFT JOIN pg_class cl2 ON cl2.oid = con.confrelid "
        "LEFT JOIN pg_namespace n2 ON n2.oid = cl2.relnamespace "
        "WHERE n.nspname = :s AND cl.relname = :t"
    )
    out = {"pk": [], "uniques": [], "fks": [], "checks": []}
    with engine.connect() as conn:
        for row in conn.execute(sql, {"s": schema, "t": table}).mappings():
            cols = list(row["cols"] or [])
            if row["contype"] == "p":
                out["pk"] = cols
            elif row["contype"] == "u":
                out["uniques"].append(cols)
            elif row["contype"] == "f":
                out["fks"].append({
                    "columns": cols,
                    "ref_schema": row["ref_schema"],
                    "ref_table": row["ref_table"],
                    "ref_columns": list(row["ref_cols"] or []),
                })
            elif row["contype"] == "c":
                out["checks"].append(str(row["def"]))
    return out


def _read_constraints_infoschema(engine: Engine, schema: str, table: str) -> dict:
    out = {"pk": [], "uniques": [], "fks": [], "checks": []}
    params = {"s": schema, "t": table}
    with engine.connect() as conn:
        pk_uni = conn.execute(text(
            "SELECT tc.constraint_type AS ctype, tc.constraint_name AS cname, "
            "       kcu.column_name AS col, kcu.ordinal_position AS pos "
            "FROM information_schema.table_constraints tc "
            "JOIN information_schema.key_column_usage kcu "
            "  ON kcu.constraint_name = tc.constraint_name "
            " AND kcu.constraint_schema = tc.constraint_schema "
            "WHERE tc.table_schema = :s AND tc.table_name = :t "
            "  AND tc.constraint_type IN ('PRIMARY KEY', 'UNIQUE') "
            "ORDER BY tc.constraint_name, kcu.ordinal_position"
        ), params).mappings().all()
        by_con: dict[str, dict] = {}
        for r in pk_uni:
            entry = by_con.setdefault(r["cname"], {"type": r["ctype"], "cols": []})
            entry["cols"].append(r["col"])
        for entry in by_con.values():
            if entry["type"] == "PRIMARY KEY":
                out["pk"] = entry["cols"]
            else:
                out["uniques"].append(entry["cols"])
        for r in conn.execute(text(
            "SELECT kcu.column_name AS col, kcu.ordinal_position AS pos, "
            "       ccu.table_schema AS ref_schema, ccu.table_name AS ref_table, "
            "       ccu.column_name AS ref_col, tc.constraint_name AS cname "
            "FROM information_schema.table_constraints tc "
            "JOIN information_schema.key_column_usage kcu "
            "  ON kcu.constraint_name = tc.constraint_name "
            " AND kcu.constraint_schema = tc.constraint_schema "
            "JOIN information_schema.constraint_column_usage ccu "
            "  ON ccu.constraint_name = tc.constraint_name "
            " AND ccu.constraint_schema = tc.constraint_schema "
            "WHERE tc.constraint_type = 'FOREIGN KEY' "
            "  AND tc.table_schema = :s AND tc.table_name = :t "
            "ORDER BY tc.constraint_name, kcu.ordinal_position"
        ), params).mappings():
            fk = next((f for f in out["fks"] if f.get("_cname") == r["cname"]), None)
            if fk is None:
                fk = {"_cname": r["cname"], "columns": [], "ref_schema": r["ref_schema"],
                      "ref_table": r["ref_table"], "ref_columns": []}
                out["fks"].append(fk)
            fk["columns"].append(r["col"])
            fk["ref_columns"].append(r["ref_col"])
        for fk in out["fks"]:
            fk.pop("_cname", None)
        for r in conn.execute(text(
            "SELECT cc.check_clause AS clause "
            "FROM information_schema.table_constraints tc "
            "JOIN information_schema.check_constraints cc "
            "  ON cc.constraint_name = tc.constraint_name "
            " AND cc.constraint_schema = tc.constraint_schema "
            "WHERE tc.table_schema = :s AND tc.table_name = :t "
            "  AND tc.constraint_type = 'CHECK'"
        ), params).mappings():
            clause = str(r["clause"])
            # information_schema плодит служебные "col IS NOT NULL" — их не тащим.
            if "IS NOT NULL" not in clause.upper():
                out["checks"].append(f"CHECK ({clause})")
    return out


def read_distribution(engine: Engine, schema: str, table: str) -> list[str]:
    """Ключ распределения Greenplum из gp_distribution_policy. [] если не GPDB
    или распределение случайное/реплицированное."""
    validate_identifier(schema, "schema")
    validate_identifier(table, "table")
    # GPDB 6+: колонка distkey (int2vector). GPDB 5: attrnums (smallint[]).
    variants = (
        "SELECT att.attname AS col "
        "FROM gp_distribution_policy p "
        "JOIN pg_class c ON c.oid = p.localoid "
        "JOIN pg_namespace n ON n.oid = c.relnamespace "
        "JOIN unnest(p.distkey) WITH ORDINALITY k(a, o) ON true "
        "JOIN pg_attribute att ON att.attrelid = c.oid AND att.attnum = k.a "
        "WHERE n.nspname = :s AND c.relname = :t ORDER BY k.o",
        "SELECT att.attname AS col "
        "FROM gp_distribution_policy p "
        "JOIN pg_class c ON c.oid = p.localoid "
        "JOIN pg_namespace n ON n.oid = c.relnamespace "
        "JOIN unnest(p.attrnums) WITH ORDINALITY k(a, o) ON true "
        "JOIN pg_attribute att ON att.attrelid = c.oid AND att.attnum = k.a "
        "WHERE n.nspname = :s AND c.relname = :t ORDER BY k.o",
    )
    for sql in variants:
        try:
            with engine.connect() as conn:
                rows = conn.execute(text(sql), {"s": schema, "t": table}).scalars().all()
            return [str(r) for r in rows]
        except Exception:  # noqa: BLE001 — пробуем следующий вариант / не GPDB
            continue
    return []


def read_partition_keys(engine: Engine, schema: str, table: str) -> list[str]:
    """Колонки партиционирования. GPDB classic (pg_partition_columns) → PG native
    (pg_partitioned_table). [] если не партиционировано / представление недоступно."""
    validate_identifier(schema, "schema")
    validate_identifier(table, "table")
    # GPDB classic partitioning.
    try:
        with engine.connect() as conn:
            rows = conn.execute(text(
                "SELECT columnname FROM pg_partition_columns "
                "WHERE schemaname = :s AND tablename = :t "
                "ORDER BY position_in_partition_key"
            ), {"s": schema, "t": table}).scalars().all()
        if rows:
            return [str(r) for r in rows]
    except Exception:  # noqa: BLE001
        pass
    # PG native declarative partitioning (GPDB 7 / Postgres).
    try:
        with engine.connect() as conn:
            cols = conn.execute(text(
                "SELECT a.attname "
                "FROM pg_partitioned_table pt "
                "JOIN pg_class c ON c.oid = pt.partrelid "
                "JOIN pg_namespace n ON n.oid = c.relnamespace "
                "JOIN unnest(pt.partattrs) WITH ORDINALITY k(a, o) ON true "
                "JOIN pg_attribute a ON a.attrelid = c.oid AND a.attnum = k.a "
                "WHERE n.nspname = :s AND c.relname = :t ORDER BY k.o"
            ), {"s": schema, "t": table}).scalars().all()
        return [str(c) for c in cols]
    except Exception:  # noqa: BLE001
        return []


In [ ]:
%%writefile generated/agc_profiler/stats_reader.py
"""Чтение статистики из pg_stats — это чтение каталога, а НЕ скан данных.

Один запрос на схему по всем нужным таблицам. Читаем:
null_frac, n_distinct, avg_width, most_common_vals, most_common_freqs, histogram_bounds.

Про n_distinct: положительное = абсолютное число distinct; отрицательное = доля
от числа строк. Знак сохраняем как есть — относительная форма удобно масштабируется
генератором.

Если по таблице pg_stats пуст/устарел — вызывающий код логирует предупреждение и
может точечно досчитать null_frac/n_distinct ТОЛЬКО по нужным колонкам через
лёгкие агрегаты (recompute_missing). По всем колонкам подряд так не делаем.
"""
from __future__ import annotations

from sqlalchemy import text
from sqlalchemy.engine import Engine

from agc_common import get_logger, quote_ident, validate_identifier

log = get_logger("profiler.stats")


def read_pg_stats(engine: Engine, schema: str, tables: list[str]) -> dict[str, dict[str, dict]]:
    """{table: {column: {null_frac, n_distinct, avg_width, mcv, mcf, histogram}}}.

    most_common_vals / histogram_bounds приводим к тексту (::text) и парсим как
    литерал массива — иначе anyarray не десериализуется драйвером надёжно.
    """
    validate_identifier(schema, "schema")
    for t in tables:
        validate_identifier(t, "table")
    sql = text(
        "SELECT tablename, attname, null_frac, n_distinct, avg_width, "
        "       most_common_vals::text AS mcv, most_common_freqs, "
        "       histogram_bounds::text AS histogram "
        "FROM pg_stats WHERE schemaname = :s AND tablename = ANY(:tables)"
    )
    out: dict[str, dict[str, dict]] = {t: {} for t in tables}
    with engine.connect() as conn:
        for row in conn.execute(sql, {"s": schema, "tables": list(tables)}).mappings():
            out.setdefault(row["tablename"], {})[row["attname"]] = {
                "null_frac": float(row["null_frac"] or 0.0),
                "n_distinct": float(row["n_distinct"]) if row["n_distinct"] is not None else None,
                "avg_width": int(row["avg_width"]) if row["avg_width"] is not None else None,
                "mcv": parse_pg_array(row["mcv"]),
                "mcf": [float(x) for x in (row["most_common_freqs"] or [])],
                "histogram": parse_pg_array(row["histogram"]),
            }
    return out


def recompute_missing(engine: Engine, schema: str, table: str, columns: list[str]) -> dict[str, dict]:
    """Точечный досчёт null_frac и n_distinct для указанных колонок через лёгкие
    агрегаты. Внимание: это СКАН таблицы — вызывать только для колонок без
    pg_stats и только по явному флагу. Логируем предупреждение."""
    validate_identifier(schema, "schema")
    validate_identifier(table, "table")
    if not columns:
        return {}
    log.warning("pg_stats пуст по %s.%s — досчитываю %d колонок агрегатами (скан!): %s",
                schema, table, len(columns), ", ".join(columns))
    parts = ["COUNT(*) AS _total"]
    for i, col in enumerate(columns):
        validate_identifier(col, "column")
        q = quote_ident(col)
        parts.append(f"COUNT(DISTINCT {q}) AS d{i}")
        parts.append(f"COUNT(*) FILTER (WHERE {q} IS NULL) AS nn{i}")
    sql = f'SELECT {", ".join(parts)} FROM "{schema}"."{table}"'
    with engine.connect() as conn:
        row = conn.execute(text(sql)).mappings().first()
    total = int(row["_total"]) or 1
    out: dict[str, dict] = {}
    for i, col in enumerate(columns):
        distinct = int(row[f"d{i}"])
        nulls = int(row[f"nn{i}"])
        out[col] = {
            "null_frac": nulls / total,
            "n_distinct": float(distinct),  # абсолютное; знак положительный
            "avg_width": None, "mcv": [], "mcf": [], "histogram": [],
        }
    return out


def parse_pg_array(literal: str | None) -> list[str]:
    """Парсер текстового литерала PG-массива: {a,b,"c,d",NULL} -> ['a','b','c,d', None].

    Достаточен для most_common_vals/histogram_bounds (строки/числа/даты).
    NULL внутри массива -> None.
    """
    if not literal:
        return []
    s = literal.strip()
    if not (s.startswith("{") and s.endswith("}")):
        return []
    s = s[1:-1]
    out: list = []
    i, n = 0, len(s)
    while i < n:
        if s[i] == '"':
            i += 1
            buf = []
            while i < n:
                c = s[i]
                if c == "\\" and i + 1 < n:
                    buf.append(s[i + 1]); i += 2; continue
                if c == '"':
                    i += 1; break
                buf.append(c); i += 1
            out.append("".join(buf))
            if i < n and s[i] == ",":
                i += 1
        else:
            j = s.find(",", i)
            if j == -1:
                j = n
            token = s[i:j].strip()
            out.append(None if token == "NULL" else token)
            i = j + 1
    return out


In [ ]:
%%writefile generated/agc_profiler/sampler.py
"""Адаптивный сэмплер — ТОЛЬКО для инспекции форматов/длин/доли NULL
чувствительных колонок. НЕ для кардинальности и НЕ для распределения значений
(их берём из pg_stats/каталога — см. profile_builder).

Оценка числа строк est берётся из планировщика (EXPLAIN, без выполнения)
или из reltuples. Стратегия:
  - 0 < est <= порога → ORDER BY random() LIMIT :n  (точная выборка, дёшево на малых);
  - est > порога      → WHERE random() < p LIMIT :n, p=min(1, oversample*n/est) (без сортировки);
  - est <= 0 / None    → считаем объект большим (assumed rows), берём p-стратегию.
"""
from __future__ import annotations

import json

from sqlalchemy import text
from sqlalchemy.engine import Engine

from agc_common import get_logger, quote_ident, validate_identifier

log = get_logger("profiler.sampler")

SAMPLE_SORT_THRESHOLD = 2_000_000
SAMPLE_OVERSAMPLE = 2.0
SAMPLE_ASSUMED_ROWS_UNKNOWN = 10_000_000


def estimate_row_count(engine: Engine, schema: str, table: str) -> int | None:
    """Оценка числа строк через планировщик (EXPLAIN FORMAT JSON, без скана).
    Работает и для вью (планировщик их раскрывает). None — оценить не удалось."""
    validate_identifier(schema, "schema")
    validate_identifier(table, "table")
    sql = f'EXPLAIN (FORMAT JSON) SELECT * FROM "{schema}"."{table}"'
    try:
        with engine.connect() as conn:
            raw = conn.execute(text(sql)).scalar()
        data = json.loads(raw) if isinstance(raw, str) else raw
        rows = int(data[0]["Plan"]["Plan Rows"])
        return rows if rows > 0 else None
    except Exception as exc:  # noqa: BLE001
        log.info("estimate_row_count(%s.%s) не удалась: %s", schema, table, exc)
        return None


def build_sample_sql(schema: str, table: str, cols_expr: str, n: int, est: int | None) -> str:
    """SQL адаптивного сэмпла по оценке est.

    ВАЖНАЯ ОГОВОРКА (смещение):
    `WHERE random() < p LIMIT n` обрывает скан по достижении n строк, то есть
    выбирает «начало физического скана», а не равномерную выборку по всей таблице.
    Для GPDB (партиции, append-optimized, порядок по времени вставки) это даёт
    смещение по СОДЕРЖИМОМУ. Поэтому из такого сэмпла берём ТОЛЬКО то, что от
    порядка не зависит: долю NULL, длины, форматы/маски. Распределение значений и
    кардинальность — из pg_stats/каталога, где смещения нет.

    Для длинных текстовых колонок в cols_expr передавайте length(col), а не сам
    col — не тащим реальные длинные значения без нужды.
    """
    validate_identifier(schema, "schema")
    validate_identifier(table, "table")
    base = f'SELECT {cols_expr} FROM "{schema}"."{table}"'
    if est is not None and 0 < est <= SAMPLE_SORT_THRESHOLD:
        return base + " ORDER BY random() LIMIT :n"
    denom = est if (est and est > 0) else SAMPLE_ASSUMED_ROWS_UNKNOWN
    p = min(1.0, SAMPLE_OVERSAMPLE * float(n) / float(denom))
    return base + f" WHERE random() < {p:.10g} LIMIT :n"


# Типы, для которых берём length(col) вместо самого значения (не тащим длинный текст).
_LONG_TEXT = ("text", "character varying", "varchar", "character", "char", "bytea", "json", "jsonb")


def _is_long_text(pg_type: str) -> bool:
    t = (pg_type or "").lower()
    return any(t.startswith(p) for p in _LONG_TEXT)


def sample_columns(
    engine: Engine,
    schema: str,
    table: str,
    columns: list[dict],
    n: int = 20_000,
    *,
    est: int | None = None,
    timeout_ms: int | None = None,
) -> dict[str, list]:
    """Возвращает {col: [значения-или-длины]} для указанных колонок.

    columns — список dict с ключами name/pg_type. Для длинного текста берём
    length(col) (число), иначе само значение (для инспекции формата коротких
    полей: телефон/email/счёт). Только порядко-независимые признаки пригодны.
    """
    if not columns:
        return {}
    if est is None:
        est = estimate_row_count(engine, schema, table)
    select_parts, out_names = [], []
    for col in columns:
        name = validate_identifier(col["name"], "column")
        if _is_long_text(col.get("pg_type", "")):
            select_parts.append(f'length({quote_ident(name)}) AS {quote_ident(name)}')
        else:
            select_parts.append(quote_ident(name))
        out_names.append(name)
    sql = build_sample_sql(schema, table, ", ".join(select_parts), n, est)
    strategy = "sort_random" if "ORDER BY random()" in sql else "random_filter"
    log.info("sample %s.%s: strategy=%s est=%s n=%d cols=%d",
             schema, table, strategy, est, n, len(out_names))
    result: dict[str, list] = {name: [] for name in out_names}
    with engine.connect() as conn:
        if timeout_ms:
            conn.execute(text(f"SET statement_timeout = {int(timeout_ms)}"))
        for row in conn.execute(text(sql), {"n": int(n)}).mappings():
            for name in out_names:
                result[name].append(row[name])
    return result


In [ ]:
%%writefile generated/agc_profiler/classifier.py
"""Авто-эвристики классификации колонок — они только ПРЕДЛАГАЮТ класс.

Финальное решение принимает policy (см. policy.py): всё, что не одобрено явно,
синтезируется. Классификатор не может пометить колонку categorical_keep — это
делает только whitelist в policy-файле (граница безопасности).

Классы: categorical_candidate, sensitive_numeric, pii, key, datetime, sensitive.
Для pii дополнительно предлагаем имя генератора (full_name/email/phone/...).
"""
from __future__ import annotations

import re

# Регэкспы форматов значений (проверяются по сэмплу коротких полей).
_RE_EMAIL = re.compile(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")
_RE_PHONE = re.compile(r"^[+()\-\s\d]{7,20}$")
_RE_INN = re.compile(r"^\d{10}$|^\d{12}$")
_RE_SNILS = re.compile(r"^\d{3}-\d{3}-\d{3} \d{2}$|^\d{11}$")
_RE_ACCOUNT = re.compile(r"^\d{20}$")

# Подсказки по ИМЕНИ колонки → (класс, генератор).
_NAME_HINTS = (
    (re.compile(r"e?mail", re.I), "pii", "email"),
    (re.compile(r"phone|tel|mobile|телефон", re.I), "pii", "phone"),
    (re.compile(r"\binn\b|_inn|инн", re.I), "pii", "inn"),
    (re.compile(r"snils|снилс", re.I), "pii", "snils"),
    (re.compile(r"account|schet|счет|счёт|\bacc\b|iban|card", re.I), "pii", "account"),
    (re.compile(r"passport|паспорт", re.I), "pii", "passport"),
    (re.compile(r"fio|full_?name|fam|surname|lastname|firstname|имя|фамилия|фио", re.I), "pii", "full_name"),
    (re.compile(r"\bname\b|_name$|наимен", re.I), "pii", "full_name"),
    (re.compile(r"address|addr|адрес", re.I), "pii", "address"),
    (re.compile(r"company|org|orgname|организац|компан", re.I), "pii", "company"),
    (re.compile(r"birth|dob|рожд", re.I), "datetime", None),
)

_NUMERIC_TYPES = ("smallint", "integer", "bigint", "numeric", "decimal",
                  "real", "double precision", "money")
_DATETIME_TYPES = ("date", "timestamp", "time")
# Порог кардинальности для кандидата в категориальные (по абсолютному n_distinct).
CATEGORICAL_MAX_DISTINCT = 50


def _base_type(pg_type: str) -> str:
    return (pg_type or "").split("(")[0].strip().lower()


def _looks_like(values: list, regex: re.Pattern) -> bool:
    """>=80% непустых сэмпл-значений матчат regex."""
    vals = [str(v) for v in values if v is not None and str(v) != ""]
    if len(vals) < 3:
        return False
    hits = sum(1 for v in vals if regex.match(v))
    return hits / len(vals) >= 0.8


def propose(
    name: str,
    pg_type: str,
    *,
    is_pk: bool = False,
    is_fk: bool = False,
    n_distinct: float | None = None,
    sample_values: list | None = None,
) -> tuple[str, str | None]:
    """Возвращает (предложенный_класс, генератор|None)."""
    base = _base_type(pg_type)

    if is_pk or is_fk:
        return "key", None

    # Формат значений из сэмпла (только короткие поля попадают сюда со значениями).
    sv = sample_values or []
    if _looks_like(sv, _RE_EMAIL):
        return "pii", "email"
    if _looks_like(sv, _RE_ACCOUNT):
        return "pii", "account"
    if _looks_like(sv, _RE_INN):
        return "pii", "inn"
    if _looks_like(sv, _RE_SNILS):
        return "pii", "snils"

    # Подсказки по имени.
    for regex, klass, gen in _NAME_HINTS:
        if regex.search(name):
            return klass, gen

    if base in _DATETIME_TYPES:
        return "datetime", None

    # Текст с низкой кардинальностью → КАНДИДАТ в категориальные (не keep!).
    if base in ("text", "character varying", "varchar", "character", "char"):
        if n_distinct is not None and 0 < n_distinct <= CATEGORICAL_MAX_DISTINCT:
            return "categorical_candidate", None
        return "sensitive", "text"

    if base in _NUMERIC_TYPES:
        # Маленькая кардинальность у числа — тоже кандидат в категориальные (коды/флаги).
        if n_distinct is not None and 0 < n_distinct <= CATEGORICAL_MAX_DISTINCT:
            return "categorical_candidate", None
        return "sensitive_numeric", None

    if base in ("boolean",):
        return "categorical_candidate", None

    return "sensitive", "text"


In [ ]:
%%writefile generated/agc_profiler/policy.py
"""Policy — whitelist чувствительности (граница безопасности).

По умолчанию КАЖДАЯ колонка считается чувствительной и будет синтезироваться.
Реальные значения (most_common_vals/histogram_bounds) попадают в профиль ТОЛЬКО
для колонок, явно помеченных в policy как categorical_keep. Никакая авто-эвристика
не может выдать categorical_keep — это делает только человек в YAML.

Формат policy-файла (YAML):

    version: 1
    columns:
      "schema.table.col":  categorical_keep          # короткая форма
      "schema.table.name": {class: pii, generator: full_name}
      "schema.table.amt":  {class: sensitive_numeric, dist: lognormal, avg_hint: "~5e4"}
    tables:
      "schema.table":
        order_groups: [["created_at", "updated_at"]]  # даты: created <= updated

Разрешённые классы в policy: categorical_keep, sensitive_numeric, pii, key,
datetime, sensitive.
"""
from __future__ import annotations

from pathlib import Path

import yaml

from agc_common import get_logger

log = get_logger("profiler.policy")

VALID_CLASSES = {
    "categorical_keep", "sensitive_numeric", "pii", "key", "datetime", "sensitive",
}
# Классы, которые НЕЛЬЗЯ вывести автоматически — только явным whitelist.
WHITELIST_ONLY = {"categorical_keep"}


class Policy:
    def __init__(self, columns: dict, tables: dict):
        self._columns = columns          # "s.t.c" -> {"class":..., extras...}
        self._tables = tables            # "s.t"   -> {"order_groups": [...]}

    @classmethod
    def load(cls, path: str | Path | None) -> "Policy":
        if not path:
            log.info("Policy-файл не задан — все колонки трактуются как чувствительные.")
            return cls({}, {})
        data = yaml.safe_load(Path(path).read_text(encoding="utf-8")) or {}
        columns: dict[str, dict] = {}
        for key, val in (data.get("columns") or {}).items():
            if isinstance(val, str):
                entry = {"class": val}
            elif isinstance(val, dict):
                entry = dict(val)
            else:
                raise ValueError(f"policy.columns[{key!r}] должен быть строкой или dict")
            klass = entry.get("class")
            if klass not in VALID_CLASSES:
                raise ValueError(f"policy.columns[{key!r}].class={klass!r} не из {VALID_CLASSES}")
            columns[key] = entry
        tables = dict(data.get("tables") or {})
        log.info("Policy загружена: %d правил по колонкам, %d по таблицам.",
                 len(columns), len(tables))
        return cls(columns, tables)

    def resolve(self, schema: str, table: str, column: str, proposed: str,
                proposed_gen: str | None) -> dict:
        """Возвращает финальный dict политики для колонки: {'class':..., extras}.

        Правила:
        - явное правило в policy побеждает всегда;
        - иначе берём предложенный класс, НО categorical_candidate → categorical_synth
          (реальные значения не сохраняем, синтезируем токены с сохранением
          кардинальности/долей);
        - categorical_keep недостижим без явного whitelist.
        """
        key = f"{schema}.{table}.{column}"
        if key in self._columns:
            entry = dict(self._columns[key])
            entry.setdefault("generator", proposed_gen)
            entry["source"] = "policy"
            return entry
        # Дефолт (whitelist): ничего не «кипаем».
        if proposed == "categorical_candidate":
            klass = "categorical_synth"
        elif proposed in VALID_CLASSES:
            klass = proposed
        else:
            klass = "sensitive"
        return {"class": klass, "generator": proposed_gen, "source": "auto"}

    def order_groups(self, schema: str, table: str) -> list[list[str]]:
        return list((self._tables.get(f"{schema}.{table}") or {}).get("order_groups") or [])


In [ ]:
%%writefile generated/agc_profiler/linter.py
"""Профиль-линтер — главная точка аудита утечек. Делаем строгим.

Правило: если колонка НЕ класса categorical_keep, но в её профиле оказались
реальные значения (values / most_common_vals / histogram / histogram_bounds /
реальные min/max) — это ошибка, падаем с явным сообщением ДО записи profile.json.
"""
from __future__ import annotations

# Ключи, несущие реальные значения строк. Разрешены ТОЛЬКО в categorical_keep.
FORBIDDEN_REAL_VALUE_KEYS = (
    "values", "most_common_vals", "mcv",
    "histogram", "histogram_bounds",
    "min_real", "max_real", "min", "max",
)


class ProfileLeakError(Exception):
    """Линтер нашёл реальные значения в неразрешённой колонке."""


def check_profile(profile: dict) -> None:
    """Бросает ProfileLeakError со списком нарушений. None — если чисто."""
    violations: list[str] = []
    for table in profile.get("tables", []):
        schema = table.get("schema")
        tname = table.get("table")
        for col_name, col in (table.get("columns") or {}).items():
            klass = col.get("policy")
            if klass == "categorical_keep":
                continue
            leaked = [k for k in FORBIDDEN_REAL_VALUE_KEYS
                      if k in col and col[k] not in (None, [], {}, "")]
            for k in leaked:
                violations.append(
                    f"{schema}.{tname}.{col_name}: класс '{klass}' содержит реальные "
                    f"значения в поле '{k}' (разрешено только для categorical_keep)"
                )
    if violations:
        raise ProfileLeakError(
            "Линтер профиля: обнаружены потенциальные утечки реальных значений:\n  - "
            + "\n  - ".join(violations)
        )


In [ ]:
%%writefile generated/agc_profiler/profile_builder.py
"""Сборка profile.json из каталога + pg_stats + (опц.) сэмпла форматов.

Для каждой колонки собираем ТОЛЬКО поля, разрешённые её классом:
- categorical_keep : реальные distinct-значения и частоты (единственный класс,
                     куда уходят most_common_vals);
- categorical_synth: n_distinct + null_frac + форма частот (числа), БЕЗ значений;
- sensitive_numeric: тип, precision/scale, null_frac, n_distinct, форма распределения,
                     грубый ориентир среднего (порядок величины), БЕЗ mcv/histogram/min/max;
- pii             : тип, null_frac, avg_width, генератор, unique_like, БЕЗ значений;
- key             : роль pk/fk, references, n_distinct, null_frac, fanout;
- datetime        : тип, null_frac, синтетический диапазон, order_group.
"""
from __future__ import annotations

import math
from datetime import datetime, timezone

from sqlalchemy.engine import Engine

from agc_common import get_logger
from agc_profiler import catalog_reader as cat
from agc_profiler import classifier as clf
from agc_profiler import stats_reader as st
from agc_profiler.policy import Policy
from agc_profiler.sampler import estimate_row_count, sample_columns

log = get_logger("profiler.build")

PROFILE_VERSION = 1


def _coarse_magnitude(value: float) -> str | None:
    """Огрубление до порядка величины: 48213.7 -> '~5e4'. Не реальное значение строки."""
    try:
        v = abs(float(value))
    except (TypeError, ValueError):
        return None
    if v == 0:
        return "~0"
    exp = int(math.floor(math.log10(v)))
    lead = round(v / (10 ** exp))
    if lead == 10:
        lead, exp = 1, exp + 1
    return f"~{lead}e{exp}"


def _numeric_avg_hint(policy_entry: dict, hist: list) -> str | None:
    """avg_hint: сначала из policy; иначе огрублённая медиана histogram_bounds
    (внутренняя величина, наружу уходит только порядок, не само значение)."""
    if policy_entry.get("avg_hint"):
        return str(policy_entry["avg_hint"])
    nums = []
    for x in hist or []:
        try:
            nums.append(float(x))
        except (TypeError, ValueError):
            continue
    if not nums:
        return None
    nums.sort()
    return _coarse_magnitude(nums[len(nums) // 2])


def _build_categorical_keep(stats: dict) -> dict:
    mcv, mcf = stats.get("mcv") or [], stats.get("mcf") or []
    values = [[v, round(float(f), 6)] for v, f in zip(mcv, mcf)]
    covered = sum(f for _, f in values)
    # Хвост распределения (всё, что вне most_common) — одной синтетической группой.
    if covered < 0.999 and (stats.get("null_frac") or 0) + covered < 0.999:
        values.append(["__other__", round(max(0.0, 1.0 - covered - (stats.get("null_frac") or 0)), 6)])
    return {
        "null_frac": round(stats.get("null_frac") or 0.0, 6),
        "n_distinct": stats.get("n_distinct"),
        "values": values,
    }


def _build_column(schema: str, table: str, col: dict, stats: dict, policy_entry: dict,
                  cons: dict, order_group_cols: set) -> dict:
    klass = policy_entry["class"]
    name = col["name"]
    base = {"pg_type": col["pg_type"], "policy": klass, "proposed_source": policy_entry.get("source")}
    null_frac = round(stats.get("null_frac") or 0.0, 6)
    n_distinct = stats.get("n_distinct")

    if klass == "categorical_keep":
        base.update(_build_categorical_keep(stats))
    elif klass == "categorical_synth":
        # Форму частот (числа) сохраняем, реальные метки — НЕТ.
        freqs = [round(float(f), 6) for f in (stats.get("mcf") or [])]
        base.update({"null_frac": null_frac, "n_distinct": n_distinct})
        if freqs:
            base["freqs"] = freqs
    elif klass == "sensitive_numeric":
        base.update({
            "null_frac": null_frac,
            "n_distinct": n_distinct,
            "precision": col.get("precision"),
            "scale": col.get("scale"),
            "dist": policy_entry.get("dist", "lognormal"),
        })
        hint = _numeric_avg_hint(policy_entry, stats.get("histogram"))
        if hint:
            base["avg_hint"] = hint
    elif klass == "pii":
        uniqueish = _is_unique_like(name, n_distinct, cons)
        base.update({
            "null_frac": null_frac,
            "avg_width": stats.get("avg_width"),
            "generator": policy_entry.get("generator") or "text",
            "unique_like": uniqueish,
        })
        if col.get("char_len"):
            base["char_len"] = col["char_len"]
    elif klass == "key":
        role = "pk" if name in (cons.get("pk") or []) else ("fk" if _is_fk(name, cons) else "key")
        base.update({"role": role, "null_frac": null_frac, "n_distinct": n_distinct})
        fk = _fk_for(name, cons)
        if fk:
            idx = fk["columns"].index(name)
            ref_col = fk["ref_columns"][idx] if idx < len(fk["ref_columns"]) else None
            base["references"] = {"schema": fk.get("ref_schema"),
                                  "table": fk.get("ref_table"), "column": ref_col}
            base["fanout"] = {"dist": "uniform"}  # форма fan-out по умолчанию равномерная
    elif klass == "datetime":
        base.update({"null_frac": null_frac})
        # Диапазон СИНТЕТИЧЕСКИЙ (не реальные min/max): дефолт — последние ~7 лет.
        base["range"] = policy_entry.get("range", {"start": "2018-01-01", "end": "2024-12-31"})
        if name in order_group_cols:
            base["order_group"] = True
    else:  # sensitive (generic)
        base.update({
            "null_frac": null_frac,
            "avg_width": stats.get("avg_width"),
            "generator": policy_entry.get("generator") or "text",
        })
    return base


def _is_fk(name: str, cons: dict) -> bool:
    return any(name in (fk.get("columns") or []) for fk in cons.get("fks") or [])


def _fk_for(name: str, cons: dict) -> dict | None:
    for fk in cons.get("fks") or []:
        if name in (fk.get("columns") or []):
            return fk
    return None


def _is_unique_like(name: str, n_distinct: float | None, cons: dict) -> bool:
    if name in (cons.get("pk") or []):
        return True
    if any(name in u for u in cons.get("uniques") or []):
        return True
    if n_distinct is not None and n_distinct < 0 and n_distinct <= -0.9:
        return True
    return False


def build_table_profile(engine: Engine, schema: str, table: str, policy: Policy,
                        table_stats: dict, *, sample: bool = False,
                        recompute_missing: bool = False) -> dict:
    meta = cat.read_table_meta(engine, schema, table)
    columns = cat.read_columns(engine, schema, table)
    cons = cat.read_constraints(engine, schema, table)
    dist_key = cat.read_distribution(engine, schema, table)
    part_keys = cat.read_partition_keys(engine, schema, table)

    pk_set = set(cons.get("pk") or [])
    fk_cols = {c for fk in cons.get("fks") or [] for c in (fk.get("columns") or [])}

    # Досчёт статистики по колонкам без pg_stats (по флагу — это скан).
    if recompute_missing:
        missing = [c["name"] for c in columns if c["name"] not in table_stats]
        if missing:
            table_stats = {**table_stats,
                           **st.recompute_missing(engine, schema, table, missing)}
    else:
        missing = [c["name"] for c in columns if c["name"] not in table_stats]
        if missing:
            log.warning("Нет pg_stats для %d колонок %s.%s (ANALYZE не делался?): %s. "
                        "Запустите с --recompute-missing для точечного досчёта.",
                        len(missing), schema, table, ", ".join(missing[:8]))

    # Сэмпл форматов — только для колонок-кандидатов в pii/sensitive (короткие поля).
    samples: dict[str, list] = {}
    if sample:
        need = [c for c in columns
                if c["name"] not in pk_set and c["name"] not in fk_cols
                and clf._base_type(c["pg_type"]) in
                ("text", "character varying", "varchar", "character", "char")]
        if need:
            try:
                samples = sample_columns(engine, schema, table, need,
                                         est=meta.get("reltuples") or None)
            except Exception as exc:  # noqa: BLE001
                log.warning("Сэмпл %s.%s не удался: %s", schema, table, exc)

    order_group_cols = {c for grp in policy.order_groups(schema, table) for c in grp}

    out_columns: dict[str, dict] = {}
    for col in columns:
        name = col["name"]
        stats = table_stats.get(name, {})
        proposed, gen = clf.propose(
            name, col["pg_type"],
            is_pk=name in pk_set, is_fk=name in fk_cols,
            n_distinct=stats.get("n_distinct"),
            sample_values=samples.get(name),
        )
        policy_entry = policy.resolve(schema, table, name, proposed, gen)
        out_columns[name] = _build_column(schema, table, col, stats, policy_entry,
                                           cons, order_group_cols)

    profile = {
        "schema": schema,
        "table": table,
        "relkind": meta["relkind"],
        "is_view": meta["is_view"],
        "storage": meta["storage"],
        "distributed_by": dist_key,
        "partitioned_by": part_keys,
        "row_count": {"value": meta["reltuples"], "estimated": meta["row_count_estimated"]},
        "constraints": {
            "pk": cons.get("pk") or [],
            "uniques": cons.get("uniques") or [],
            "fks": [{k: v for k, v in fk.items() if not k.startswith("_")}
                    for fk in cons.get("fks") or []],
            "checks": cons.get("checks") or [],
        },
        "defaults": {c["name"]: c["default"] for c in columns if c["default"]},
        "not_null": [c["name"] for c in columns if not c["nullable"]],
        "columns": out_columns,
    }
    log.info("Профиль %s.%s: %d колонок, storage=%s, dist=%s, part=%s, rows~%s",
             schema, table, len(out_columns), meta["storage"], dist_key or "-",
             part_keys or "-", meta["reltuples"])
    return profile


def build_profile(engine: Engine, tables: list[tuple[str, str]], policy: Policy,
                  *, sample: bool = False, recompute_missing: bool = False) -> dict:
    """Собирает полный профиль по списку (schema, table). pg_stats читаем пачкой на схему."""
    by_schema: dict[str, list[str]] = {}
    for schema, table in tables:
        by_schema.setdefault(schema, []).append(table)

    stats_cache: dict[str, dict[str, dict]] = {}
    for schema, tnames in by_schema.items():
        try:
            stats_cache[schema] = st.read_pg_stats(engine, schema, tnames)
        except Exception as exc:  # noqa: BLE001
            log.warning("pg_stats по схеме %s не прочитан: %s", schema, exc)
            stats_cache[schema] = {t: {} for t in tnames}

    table_profiles = []
    for schema, table in tables:
        table_stats = stats_cache.get(schema, {}).get(table, {})
        table_profiles.append(build_table_profile(
            engine, schema, table, policy, table_stats,
            sample=sample, recompute_missing=recompute_missing,
        ))

    return {
        "profile_version": PROFILE_VERSION,
        "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "source_dialect": "greenplum",
        "note": "Синтетический профиль: реальные значения только в categorical_keep.",
        "tables": table_profiles,
    }


In [ ]:
%%writefile generated/agc_profiler/cli.py
"""CLI программы 1 (profiler). Закрытый контур.

Пример:
    python -m agc_profiler.cli --tables "public.tasks,public.clients" \
        --policy policy.yaml --out profile.json --sample

    python -m agc_profiler.cli --tables-csv tables.csv --out profile.json
"""
from __future__ import annotations

import argparse
import csv
import json
import sys
from pathlib import Path

from agc_common import get_logger
from agc_profiler.db import make_engine
from agc_profiler.linter import check_profile
from agc_profiler.policy import Policy
from agc_profiler.profile_builder import build_profile

log = get_logger("profiler.cli")


def parse_tables_arg(value: str) -> list[tuple[str, str]]:
    """'schema.table,schema.table2' -> [(schema, table), ...]."""
    out = []
    for item in value.split(","):
        item = item.strip()
        if not item:
            continue
        if "." not in item:
            raise ValueError(f"Ожидался формат schema.table, получено: {item!r}")
        schema, table = item.split(".", 1)
        out.append((schema.strip(), table.strip()))
    return out


def parse_tables_csv(path: str | Path) -> list[tuple[str, str]]:
    """CSV со списком таблиц. Дефолт-колонки: schema,table. Поддерживаем также
    schema_name,table_name (как tables_list.csv проекта)."""
    rows: list[tuple[str, str]] = []
    with open(path, newline="", encoding="utf-8") as fh:
        reader = csv.DictReader(fh)
        cols = {c.lower(): c for c in (reader.fieldnames or [])}
        sc = cols.get("schema") or cols.get("schema_name")
        tc = cols.get("table") or cols.get("table_name")
        if not sc or not tc:
            raise ValueError("CSV должен содержать колонки schema,table (или schema_name,table_name)")
        for r in reader:
            schema, table = (r[sc] or "").strip(), (r[tc] or "").strip()
            if schema and table:
                rows.append((schema, table))
    return rows


def build_arg_parser() -> argparse.ArgumentParser:
    p = argparse.ArgumentParser(prog="agc_profiler", description="Profiler (закрытый контур)")
    src = p.add_mutually_exclusive_group(required=True)
    src.add_argument("--tables", help="schema.table через запятую")
    src.add_argument("--tables-csv", help="CSV со списком таблиц (колонки schema,table)")
    p.add_argument("--policy", help="YAML policy-файл (whitelist чувствительности)")
    p.add_argument("--out", default="profile.json", help="куда писать профиль")
    p.add_argument("--dsn", help="DSN БД (иначе AGC_DB_DSN/DB_DSN/db_config.json/проект)")
    p.add_argument("--db-config", help="db_config.json с параметрами подключения")
    p.add_argument("--sample", action="store_true",
                   help="инспектировать форматы коротких текстовых полей адаптивным сэмплом")
    p.add_argument("--recompute-missing", action="store_true",
                   help="досчитать null_frac/n_distinct агрегатами там, где пуст pg_stats (СКАН!)")
    p.add_argument("--statement-timeout-ms", type=int, default=600_000)
    return p


def main(argv: list[str] | None = None) -> int:
    args = build_arg_parser().parse_args(argv)
    tables = parse_tables_csv(args.tables_csv) if args.tables_csv else parse_tables_arg(args.tables)
    if not tables:
        log.error("Список таблиц пуст."); return 2
    log.info("К обработке %d таблиц: %s", len(tables),
             ", ".join(f"{s}.{t}" for s, t in tables[:10]))

    engine = make_engine(args.dsn, args.db_config, args.statement_timeout_ms)
    policy = Policy.load(args.policy)
    profile = build_profile(engine, tables, policy,
                            sample=args.sample, recompute_missing=args.recompute_missing)

    # Линтер — строгая проверка перед записью (главная точка аудита утечек).
    check_profile(profile)
    log.info("Линтер пройден: реальные значения только в categorical_keep.")

    Path(args.out).write_text(
        json.dumps(profile, ensure_ascii=False, indent=2), encoding="utf-8")
    log.info("Профиль записан: %s (%d таблиц)", args.out, len(profile["tables"]))
    return 0


if __name__ == "__main__":
    sys.exit(main())


### Программа 2 — generator (открытый контур)

Модули: `profile_parser` · `ddl_builder` (schema.sql, топосорт по FK) · `value_generators` (значения по классам) · `key_linker` (PK/FK, масштабирование) · `writer` (CSV/INSERT) · `cli`.

In [ ]:
%%writefile generated/agc_generator/__init__.py
"""Программа 2 — generator (открытый контур).

Читает profile.json, строит schema.sql (DDL) и синтетические данные (CSV или
INSERT-ы). Лёгкий детерминированный генератор по профилю — без ML-синтезаторов
(SDV/CTGAN и т.п.), чтобы физически не воспроизводить реальные совместные
распределения и не «запоминать» реальные строки.
"""


In [ ]:
%%writefile generated/agc_generator/profile_parser.py
"""Парсер profile.json в удобные структуры для генератора."""
from __future__ import annotations

import json
from dataclasses import dataclass, field
from pathlib import Path


@dataclass
class Column:
    name: str
    pg_type: str
    policy: str
    raw: dict = field(default_factory=dict)

    def get(self, key, default=None):
        return self.raw.get(key, default)


@dataclass
class Table:
    schema: str
    table: str
    is_view: bool
    storage: str
    distributed_by: list
    partitioned_by: list
    row_count: int
    row_count_estimated: bool
    constraints: dict
    defaults: dict
    not_null: list
    columns: dict  # name -> Column

    @property
    def fqn(self) -> str:
        return f"{self.schema}.{self.table}"

    @property
    def pk(self) -> list:
        return self.constraints.get("pk") or []

    @property
    def fks(self) -> list:
        return self.constraints.get("fks") or []


@dataclass
class Profile:
    version: int
    tables: list  # list[Table]

    def by_fqn(self) -> dict:
        return {t.fqn: t for t in self.tables}


def load_profile(path: str | Path) -> Profile:
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    tables = []
    for t in data.get("tables", []):
        columns = {
            name: Column(name=name, pg_type=c.get("pg_type", "text"),
                         policy=c.get("policy", "sensitive"), raw=c)
            for name, c in (t.get("columns") or {}).items()
        }
        rc = t.get("row_count") or {}
        tables.append(Table(
            schema=t["schema"], table=t["table"],
            is_view=bool(t.get("is_view")), storage=t.get("storage", "heap"),
            distributed_by=t.get("distributed_by") or [],
            partitioned_by=t.get("partitioned_by") or [],
            row_count=int(rc.get("value") or 0),
            row_count_estimated=bool(rc.get("estimated", True)),
            constraints=t.get("constraints") or {},
            defaults=t.get("defaults") or {},
            not_null=t.get("not_null") or [],
            columns=columns,
        ))
    return Profile(version=int(data.get("profile_version", 1)), tables=tables)


In [ ]:
%%writefile generated/agc_generator/ddl_builder.py
"""DDL-билдер: CREATE TABLE из профиля (не завися от pg_dump).

Воспроизводим типы, PK/UNIQUE/NOT NULL/FK/CHECK, DEFAULT. По умолчанию тестовая
БД — всё heap, одиночный сегмент (DISTRIBUTED BY / партиции опускаем). Флаг
keep_gpdb сохраняет GPDB-специфику (DISTRIBUTED BY, комментарий о партициях).
Внешние таблицы (gpfdist/PXF) → обычные таблицы-заглушки.

Таблицы упорядочиваем топологически по FK, чтобы родители создавались раньше детей.
FK на таблицы вне профиля опускаем (не на что ссылаться), но значения всё равно
резолвим из синтетического пула (см. key_linker).
"""
from __future__ import annotations

from agc_common import get_logger, quote_ident
from agc_generator.profile_parser import Profile, Table

log = get_logger("generator.ddl")


def topo_sort(tables: list[Table]) -> list[Table]:
    """Порядок создания: родитель раньше ребёнка. Циклы разрываем стабильно."""
    fqns = {t.fqn for t in tables}
    by_fqn = {t.fqn: t for t in tables}
    deps: dict[str, set] = {t.fqn: set() for t in tables}
    for t in tables:
        for fk in t.fks:
            ref = f"{fk.get('ref_schema')}.{fk.get('ref_table')}"
            if ref in fqns and ref != t.fqn:
                deps[t.fqn].add(ref)
    ordered, visited, temp = [], set(), set()

    def visit(fqn: str):
        if fqn in visited:
            return
        if fqn in temp:  # цикл — не углубляемся дальше
            return
        temp.add(fqn)
        for d in sorted(deps[fqn]):
            visit(d)
        temp.discard(fqn)
        visited.add(fqn)
        ordered.append(by_fqn[fqn])

    for fqn in sorted(by_fqn):
        visit(fqn)
    return ordered


def _column_ddl(table: Table, name: str) -> str:
    col = table.columns[name]
    parts = [f"  {quote_ident(name)} {col.pg_type}"]
    if name in table.not_null and name not in table.pk:
        parts.append("NOT NULL")
    default = table.defaults.get(name)
    # nextval(...) отбрасываем: секвенций в тестовой БД нет, PK генерируем сами.
    if default and "nextval(" not in str(default).lower():
        parts.append(f"DEFAULT {default}")
    return " ".join(parts)


def build_table_ddl(table: Table, profile_fqns: set, keep_gpdb: bool) -> str:
    lines = [_column_ddl(table, name) for name in table.columns]

    if table.pk:
        cols = ", ".join(quote_ident(c) for c in table.pk)
        lines.append(f"  PRIMARY KEY ({cols})")
    for uni in table.constraints.get("uniques") or []:
        cols = ", ".join(quote_ident(c) for c in uni)
        lines.append(f"  UNIQUE ({cols})")
    for fk in table.fks:
        ref = f"{fk.get('ref_schema')}.{fk.get('ref_table')}"
        if ref not in profile_fqns:
            log.info("FK %s.%s -> %s опущен (цель вне профиля)", table.schema, table.table, ref)
            continue
        cols = ", ".join(quote_ident(c) for c in fk.get("columns") or [])
        ref_cols = ", ".join(quote_ident(c) for c in fk.get("ref_columns") or [])
        lines.append(
            f"  FOREIGN KEY ({cols}) REFERENCES "
            f"{quote_ident(fk['ref_schema'])}.{quote_ident(fk['ref_table'])} ({ref_cols})"
        )
    for check in table.constraints.get("checks") or []:
        lines.append(f"  {check}")

    ddl = (
        f"CREATE TABLE {quote_ident(table.schema)}.{quote_ident(table.table)} (\n"
        + ",\n".join(lines) + "\n)"
    )
    if keep_gpdb and table.distributed_by:
        cols = ", ".join(quote_ident(c) for c in table.distributed_by)
        ddl += f"\nDISTRIBUTED BY ({cols})"
    elif keep_gpdb:
        ddl += "\nDISTRIBUTED RANDOMLY"
    ddl += ";"
    if keep_gpdb and table.partitioned_by:
        ddl += (f"\n-- NOTE: исходная таблица партиционирована по "
                f"{', '.join(table.partitioned_by)}; для теста партиции упрощены.")
    return ddl


def build_ddl(profile: Profile, *, keep_gpdb: bool = False) -> str:
    tables = [t for t in profile.tables if not t.is_view]
    views = [t for t in profile.tables if t.is_view]
    if views:
        log.info("Вью в профиле (%d) материализуем как обычные таблицы-заглушки: %s",
                 len(views), ", ".join(v.fqn for v in views))
    all_tables = tables + views  # вью тоже как таблицы
    profile_fqns = {t.fqn for t in all_tables}
    ordered = topo_sort(all_tables)

    schemas = sorted({t.schema for t in ordered})
    header = [
        "-- Автосгенерированный DDL тестовой БД (air-gap clone).",
        "-- Реальные данные не воспроизводятся; структура — из profile.json.",
        "",
    ]
    for s in schemas:
        header.append(f"CREATE SCHEMA IF NOT EXISTS {quote_ident(s)};")
    header.append("")
    body = [build_table_ddl(t, profile_fqns, keep_gpdb) for t in ordered]
    return "\n".join(header) + "\n\n".join(body) + "\n"


In [ ]:
%%writefile generated/agc_generator/value_generators.py
"""Генераторы значений по классу колонки.

Лёгкие, детерминированные (по seed), без ML-синтезаторов. numpy/Faker
опциональны — при их отсутствии работает stdlib-реализация (random/math).

Классы: categorical_keep, categorical_synth, sensitive_numeric, pii, datetime,
sensitive (generic). Ключи (key) обрабатывает key_linker.
"""
from __future__ import annotations

import math
import random
from datetime import date, datetime, timedelta

from agc_common import get_logger

log = get_logger("generator.values")

try:  # опционально — красивее pii; при отсутствии работает встроенный фолбэк
    from faker import Faker  # type: ignore
    _FAKER = Faker("ru_RU")
except Exception:  # noqa: BLE001
    _FAKER = None

# --- встроенные словари для pii-фолбэка (без внешних зависимостей) ---
_FIRST_M = ["Александр", "Дмитрий", "Максим", "Сергей", "Андрей", "Алексей", "Иван", "Кирилл"]
_FIRST_F = ["Анна", "Елена", "Ольга", "Наталья", "Мария", "Татьяна", "Ирина", "Екатерина"]
_LAST = ["Иванов", "Петров", "Смирнов", "Кузнецов", "Соколов", "Попов", "Лебедев", "Козлов"]
_STREETS = ["Ленина", "Мира", "Советская", "Гагарина", "Победы", "Садовая", "Лесная"]
_COMPANY = ["Импульс", "Вектор", "Горизонт", "Альфа", "Дельта", "Партнёр", "Синтез", "Ресурс"]
_COMPANY_FORM = ["ООО", "АО", "ПАО", "ЗАО"]


def _apply_nulls(values: list, null_frac: float, rng: random.Random) -> list:
    if null_frac <= 0:
        return values
    return [None if rng.random() < null_frac else v for v in values]


# --------------------------------------------------------------------------- #
# Категориальные                                                              #
# --------------------------------------------------------------------------- #
def gen_categorical_keep(values: list, n: int, null_frac: float, rng: random.Random) -> list:
    """Сэмпл из реальных distinct-значений по их весам — сохраняет кардинальность
    и распределение групп (для корректных GROUP BY)."""
    labels = [v[0] for v in values] or ["__empty__"]
    weights = [max(0.0, float(v[1])) for v in values] or [1.0]
    total = sum(weights) or 1.0
    weights = [w / total for w in weights]
    out = rng.choices(labels, weights=weights, k=n)
    return _apply_nulls(out, null_frac, rng)


def gen_categorical_synth(k: int, freqs: list | None, n: int, null_frac: float,
                          rng: random.Random, prefix: str = "cat") -> list:
    """K синтетических токенов, сэмпл по весам freqs (или равномерно). Реальные
    метки не используются — сохраняем только форму кардинальности/частот."""
    k = max(1, k)
    tokens = [f"{prefix}_{i:04d}" for i in range(k)]
    if freqs and len(freqs) >= 1:
        w = [max(0.0, float(x)) for x in freqs[:k]]
        while len(w) < k:  # доборем хвост маленькими весами
            w.append(min(w) if w else 1.0)
    else:
        w = [1.0] * k
    total = sum(w) or 1.0
    w = [x / total for x in w]
    out = rng.choices(tokens, weights=w, k=n)
    return _apply_nulls(out, null_frac, rng)


# --------------------------------------------------------------------------- #
# Числа                                                                       #
# --------------------------------------------------------------------------- #
def _parse_magnitude(avg_hint: str | None, default: float = 1000.0) -> float:
    if not avg_hint:
        return default
    s = str(avg_hint).lstrip("~").strip()
    try:
        return float(s)
    except ValueError:
        return default


def gen_sensitive_numeric(spec: dict, n: int, rng: random.Random) -> list:
    """Числа из указанной формы распределения. Правдоподобно для агрегатов,
    но не похоже на реальные суммы (форма и порядок — из профиля, значения — новые)."""
    dist = (spec.get("dist") or "lognormal").lower()
    scale = spec.get("scale")
    mean = _parse_magnitude(spec.get("avg_hint"), default=1000.0)
    null_frac = float(spec.get("null_frac") or 0.0)

    out: list = []
    if dist == "lognormal":
        sigma = 1.0
        mu = math.log(mean) - 0.5 * sigma * sigma if mean > 0 else 0.0
        for _ in range(n):
            out.append(math.exp(rng.gauss(mu, sigma)))
    elif dist == "normal":
        sigma = max(1.0, mean * 0.3)
        for _ in range(n):
            out.append(rng.gauss(mean, sigma))
    else:  # uniform
        lo, hi = 0.0, max(1.0, mean * 2)
        for _ in range(n):
            out.append(rng.uniform(lo, hi))

    # precision/scale и целочисленность.
    base = (spec.get("pg_type") or "").split("(")[0].strip().lower()
    is_int = base in ("smallint", "integer", "bigint")
    rounded = []
    for v in out:
        if is_int:
            rounded.append(int(round(v)))
        elif scale is not None:
            rounded.append(round(v, int(scale)))
        else:
            rounded.append(round(v, 2))
    return _apply_nulls(rounded, null_frac, rng)


# --------------------------------------------------------------------------- #
# PII                                                                         #
# --------------------------------------------------------------------------- #
def _pii_one(generator: str, rng: random.Random, seq: int) -> str:
    if _FAKER is not None:
        try:
            return _faker_value(generator, seq)
        except Exception:  # noqa: BLE001
            pass
    return _builtin_pii(generator, rng, seq)


def _faker_value(generator: str, seq: int) -> str:
    f = _FAKER
    mapping = {
        "full_name": f.name, "email": f.email, "phone": f.phone_number,
        "address": f.address, "company": f.company,
    }
    if generator in mapping:
        return str(mapping[generator]()).replace("\n", ", ")
    if generator == "inn":
        return f.numerify("##########")
    if generator == "snils":
        return f.numerify("###-###-### ##")
    if generator == "account":
        return f.numerify("#" * 20)
    if generator == "passport":
        return f.numerify("#### ######")
    return f"{generator}_{seq:06d}"


def _builtin_pii(generator: str, rng: random.Random, seq: int) -> str:
    if generator == "full_name":
        if rng.random() < 0.5:
            return f"{rng.choice(_LAST)} {rng.choice(_FIRST_M)}"
        return f"{rng.choice(_LAST)}а {rng.choice(_FIRST_F)}"
    if generator == "email":
        return f"user{seq:06d}@example.test"
    if generator == "phone":
        return "+7" + "".join(str(rng.randint(0, 9)) for _ in range(10))
    if generator == "inn":
        return "".join(str(rng.randint(0, 9)) for _ in range(10))
    if generator == "snils":
        d = "".join(str(rng.randint(0, 9)) for _ in range(11))
        return f"{d[0:3]}-{d[3:6]}-{d[6:9]} {d[9:11]}"
    if generator == "account":
        return "".join(str(rng.randint(0, 9)) for _ in range(20))
    if generator == "passport":
        return "".join(str(rng.randint(0, 9)) for _ in range(4)) + " " + \
               "".join(str(rng.randint(0, 9)) for _ in range(6))
    if generator == "address":
        return f"ул. {rng.choice(_STREETS)}, д. {rng.randint(1, 120)}, кв. {rng.randint(1, 200)}"
    if generator == "company":
        return f'{rng.choice(_COMPANY_FORM)} "{rng.choice(_COMPANY)}"'
    # generic text
    return f"{generator}_{seq:06d}"


def gen_pii(generator: str, n: int, null_frac: float, unique_like: bool,
            rng: random.Random) -> list:
    out: list = []
    seen: set = set()
    seq = 0
    while len(out) < n:
        seq += 1
        val = _pii_one(generator, rng, seq)
        if unique_like:
            if val in seen:
                val = f"{val}#{seq}"  # гарантируем уникальность
            seen.add(val)
        out.append(val)
    return _apply_nulls(out, null_frac, rng)


# --------------------------------------------------------------------------- #
# Даты                                                                        #
# --------------------------------------------------------------------------- #
def _parse_date(s: str, default: date) -> date:
    try:
        return datetime.strptime(str(s)[:10], "%Y-%m-%d").date()
    except (ValueError, TypeError):
        return default


def gen_datetime(spec: dict, n: int, rng: random.Random) -> list:
    rng_spec = spec.get("range") or {}
    start = _parse_date(rng_spec.get("start"), date(2018, 1, 1))
    end = _parse_date(rng_spec.get("end"), date(2024, 12, 31))
    span = max(1, (end - start).days)
    is_ts = "timestamp" in (spec.get("pg_type") or "").lower()
    out: list = []
    for _ in range(n):
        d = start + timedelta(days=rng.randint(0, span))
        if is_ts:
            out.append(datetime(d.year, d.month, d.day,
                                rng.randint(0, 23), rng.randint(0, 59), rng.randint(0, 59)))
        else:
            out.append(d)
    return _apply_nulls(out, float(spec.get("null_frac") or 0.0), rng)


def gen_generic(spec: dict, n: int, rng: random.Random) -> list:
    """sensitive (generic) — короткие псевдо-текстовые токены."""
    gen = spec.get("generator") or "val"
    out = [f"{gen}_{i:06d}" for i in range(n)]
    rng.shuffle(out)
    return _apply_nulls(out, float(spec.get("null_frac") or 0.0), rng)


In [ ]:
%%writefile generated/agc_generator/key_linker.py
"""Связывание ключей и ссылочная целостность + согласованное масштабирование.

Порядок: топологический по FK — родители генерируются раньше детей, чтобы дети
брали FK из уже существующих родительских ключей. ВСЕ джойны обязаны резолвиться.

Масштабирование:
- N_table = max(1, round(row_count * scale_factor));
- абсолютный n_distinct умножаем на scale_factor; относительный (-0.x) — доля от N;
- distinct зажимаем в [1, N] (нельзя 5000 уникальных клиентов в 1000 строк).

Для точной кардинальности: генерируем K токенов/ключей и сэмплируем по весам;
для «уникальных» — N различных значений.
"""
from __future__ import annotations

import random

from agc_common import get_logger
from agc_generator import value_generators as vg
from agc_generator.ddl_builder import topo_sort
from agc_generator.profile_parser import Profile, Table

log = get_logger("generator.keys")


def scaled_rowcount(row_count: int, scale: float) -> int:
    return max(1, int(round(row_count * scale)))


def scaled_distinct(n_distinct: float | None, n_rows: int, scale: float) -> int:
    """Согласованная кардинальность для ВЫСОКО-кардинальных колонок (ключи).
    n_distinct: >0 абсолютное (масштабируем на scale), <0 относительное (доля от
    n_rows). Всегда зажато в [1, n_rows]."""
    if n_distinct is None:
        return max(1, n_rows)
    if n_distinct < 0:
        d = round(-float(n_distinct) * n_rows)
    else:
        d = round(float(n_distinct) * scale)
    return max(1, min(int(d), n_rows))


# Абсолютные n_distinct, которые считаем «категориальной» кардинальностью домена
# (коды/статусы/флаги) и НЕ масштабируем вниз — иначе GROUP BY схлопнется.
CATEGORICAL_ABS_MAX = 200


def categorical_cardinality(n_distinct: float | None, freqs: list | None, n_rows: int) -> int:
    """Число distinct для categorical_synth.

    NOTE: спека предписывает 'абсолютные n_distinct * scale_factor'. Для настоящих
    категориальных доменов (статус из 5 значений) это схлопнуло бы 5→1 и сломало бы
    GROUP BY. Кардинальность enum — свойство домена, а не числа строк, поэтому здесь
    её СОХРАНЯЕМ (зажимая в [1, n_rows]), а масштабирование scale применяем только к
    высоко-кардинальным ключам (scaled_distinct). Число корзин freqs — естественный
    пол кардинальности.
    """
    floor = len(freqs) if freqs else 0
    if n_distinct is None:
        k = floor or n_rows
    elif n_distinct < 0:
        k = round(-float(n_distinct) * n_rows)
    elif n_distinct <= CATEGORICAL_ABS_MAX:
        k = int(round(n_distinct))            # маленький enum — сохраняем как есть
    else:
        k = int(round(n_distinct))            # крупный домен — тоже сохраняем распознанное число
    return max(1, min(max(k, floor), n_rows))


def _col_seed(base_seed: int, table: Table, col: str) -> int:
    return (hash((table.fqn, col)) ^ base_seed) & 0x7FFFFFFF


def _gen_pk_pool(table: Table, pk_col: str, n_rows: int, base_seed: int) -> list:
    """Суррогатные PK 1..N (для целочисленных) или синтетические токены иначе."""
    col = table.columns.get(pk_col)
    base = (col.pg_type.split("(")[0].strip().lower() if col else "bigint")
    if base in ("smallint", "integer", "bigint", "numeric", "decimal"):
        return list(range(1, n_rows + 1))
    return [f"{table.table[:8]}_{i:06d}" for i in range(1, n_rows + 1)]


def _fill_fk(child_col_spec: dict, parent_pool: list, n_rows: int, n_distinct: float | None,
             scale: float, rng: random.Random) -> list:
    """FK-значения из пула родительских ключей с учётом fan-out и доли NULL.
    Все значения гарантированно существуют у родителя → джойны резолвятся."""
    null_frac = float(child_col_spec.get("null_frac") or 0.0)
    if not parent_pool:
        return [None] * n_rows
    # Сколько различных родителей реально «используется» детьми (fan-out).
    used = scaled_distinct(n_distinct, min(len(parent_pool), n_rows), scale)
    used = max(1, min(used, len(parent_pool)))
    active_parents = rng.sample(parent_pool, used) if used < len(parent_pool) else list(parent_pool)
    out = [rng.choice(active_parents) for _ in range(n_rows)]
    if null_frac > 0:
        out = [None if rng.random() < null_frac else v for v in out]
    return out


def generate(profile: Profile, scale: float, seed: int) -> dict[str, list[dict]]:
    """Возвращает {fqn: [row_dict, ...]} со согласованными ключами и FK."""
    all_tables = list(profile.tables)
    ordered = topo_sort(all_tables)
    by_fqn = {t.fqn: t for t in all_tables}

    data: dict[str, list[dict]] = {}
    pk_pools: dict[str, dict[str, list]] = {}  # fqn -> {pk_col: pool}

    for table in ordered:
        rng = random.Random((hash(table.fqn) ^ seed) & 0x7FFFFFFF)
        n_rows = scaled_rowcount(table.row_count, scale)
        pk_cols = table.pk
        columns_out: dict[str, list] = {}

        # 1) PK-пулы (одиночный суррогатный PK — основной случай).
        single_pk = pk_cols[0] if len(pk_cols) == 1 else None
        if single_pk:
            pool = _gen_pk_pool(table, single_pk, n_rows, seed)
            columns_out[single_pk] = list(pool)
            pk_pools.setdefault(table.fqn, {})[single_pk] = pool

        # 2) FK-колонки (после того как родители сгенерированы — топо-порядок это гарантирует).
        fk_by_col: dict[str, dict] = {}
        for fk in table.fks:
            ref_fqn = f"{fk.get('ref_schema')}.{fk.get('ref_table')}"
            for i, ccol in enumerate(fk.get("columns") or []):
                ref_col = (fk.get("ref_columns") or [None] * (i + 1))[i]
                fk_by_col[ccol] = {"ref_fqn": ref_fqn, "ref_col": ref_col}

        # 3) Остальные колонки по классу.
        for name, col in table.columns.items():
            if name in columns_out:
                continue  # PK уже сгенерирован
            crng = random.Random(_col_seed(seed, table, name))
            spec = dict(col.raw)
            spec.setdefault("pg_type", col.pg_type)
            klass = col.policy

            if name in fk_by_col:
                info = fk_by_col[name]
                parent = pk_pools.get(info["ref_fqn"], {})
                parent_pool = parent.get(info["ref_col"]) or (next(iter(parent.values()), []) if parent else [])
                if not parent_pool:
                    # Родитель вне профиля — синтетический пул, чтобы FK всё же резолвился.
                    pcount = scaled_distinct(spec.get("n_distinct"), n_rows, scale)
                    parent_pool = list(range(1, pcount + 1))
                    log.info("FK %s.%s: родитель %s вне профиля — синтетический пул из %d ключей",
                             table.table, name, info["ref_fqn"], pcount)
                columns_out[name] = _fill_fk(spec, parent_pool, n_rows,
                                             spec.get("n_distinct"), scale, crng)
                continue

            if klass == "key":  # ключ без FK и не одиночный PK (напр. часть составного)
                columns_out[name] = list(range(1, n_rows + 1))
            elif klass == "categorical_keep":
                columns_out[name] = vg.gen_categorical_keep(
                    spec.get("values") or [], n_rows, float(spec.get("null_frac") or 0.0), crng)
            elif klass == "categorical_synth":
                k = categorical_cardinality(spec.get("n_distinct"), spec.get("freqs"), n_rows)
                columns_out[name] = vg.gen_categorical_synth(
                    k, spec.get("freqs"), n_rows, float(spec.get("null_frac") or 0.0),
                    crng, prefix=name[:8])
            elif klass == "sensitive_numeric":
                columns_out[name] = vg.gen_sensitive_numeric(spec, n_rows, crng)
            elif klass == "pii":
                unique_like = bool(spec.get("unique_like"))
                columns_out[name] = vg.gen_pii(
                    spec.get("generator") or "text", n_rows,
                    float(spec.get("null_frac") or 0.0), unique_like, crng)
            elif klass == "datetime":
                columns_out[name] = vg.gen_datetime(spec, n_rows, crng)
            else:  # sensitive / generic
                columns_out[name] = vg.gen_generic(spec, n_rows, crng)

        # 4) Транспонируем в строки + чиним порядок связанных дат (created <= updated).
        rows = _to_rows(table, columns_out, n_rows)
        rows = _enforce_date_order(table, rows)
        data[table.fqn] = rows
        log.info("Данные %s: %d строк, %d колонок (scale=%s)", table.fqn, n_rows,
                 len(columns_out), scale)
    return data


def _to_rows(table: Table, columns_out: dict[str, list], n_rows: int) -> list[dict]:
    names = list(table.columns.keys())
    rows = []
    for i in range(n_rows):
        rows.append({name: columns_out.get(name, [None] * n_rows)[i] for name in names})
    return rows


def _enforce_date_order(table: Table, rows: list[dict]) -> list[dict]:
    """Внутри order_group даты сортируем по возрастанию (created <= updated <= ...)."""
    groups = [name for name, col in table.columns.items() if col.get("order_group")]
    if len(groups) < 2:
        return rows
    for row in rows:
        vals = [(name, row[name]) for name in groups if row[name] is not None]
        ordered_vals = sorted(v for _, v in vals)
        j = 0
        for name in groups:
            if row[name] is not None:
                row[name] = ordered_vals[j]; j += 1
    return rows


In [ ]:
%%writefile generated/agc_generator/writer.py
"""Запись синтетических данных: CSV на таблицу или батч INSERT-ов."""
from __future__ import annotations

import csv
from datetime import date, datetime
from pathlib import Path

from agc_common import get_logger, quote_ident
from agc_generator.ddl_builder import topo_sort
from agc_generator.profile_parser import Profile

log = get_logger("generator.writer")


def _cell_csv(v) -> str:
    if v is None:
        return ""
    if isinstance(v, (datetime, date)):
        return v.isoformat(sep=" ") if isinstance(v, datetime) else v.isoformat()
    return str(v)


def write_csv(profile: Profile, data: dict[str, list[dict]], outdir: Path) -> list[Path]:
    outdir.mkdir(parents=True, exist_ok=True)
    written = []
    for table in profile.tables:
        rows = data.get(table.fqn, [])
        path = outdir / f"{table.schema}.{table.table}.csv"
        cols = list(table.columns.keys())
        with open(path, "w", newline="", encoding="utf-8") as fh:
            w = csv.writer(fh)
            w.writerow(cols)
            for row in rows:
                w.writerow([_cell_csv(row.get(c)) for c in cols])
        written.append(path)
        log.info("CSV %s: %d строк", path.name, len(rows))
    return written


def _cell_sql(v) -> str:
    if v is None:
        return "NULL"
    if isinstance(v, bool):
        return "TRUE" if v else "FALSE"
    if isinstance(v, (int, float)):
        return repr(v)
    if isinstance(v, datetime):
        return "'" + v.isoformat(sep=" ") + "'"
    if isinstance(v, date):
        return "'" + v.isoformat() + "'"
    return "'" + str(v).replace("'", "''") + "'"


def write_sql(profile: Profile, data: dict[str, list[dict]], out_path: Path,
              batch: int = 500) -> Path:
    ordered = topo_sort(list(profile.tables))  # родители раньше — FK не нарушаем при вставке
    lines = ["-- Синтетические данные (air-gap clone). INSERT-ы в порядке FK-зависимостей.", ""]
    for table in ordered:
        rows = data.get(table.fqn, [])
        if not rows:
            continue
        cols = list(table.columns.keys())
        collist = ", ".join(quote_ident(c) for c in cols)
        target = f"{quote_ident(table.schema)}.{quote_ident(table.table)}"
        for i in range(0, len(rows), batch):
            chunk = rows[i:i + batch]
            values = ",\n  ".join(
                "(" + ", ".join(_cell_sql(r.get(c)) for c in cols) + ")" for r in chunk)
            lines.append(f"INSERT INTO {target} ({collist}) VALUES\n  {values};")
        lines.append("")
    out_path.write_text("\n".join(lines), encoding="utf-8")
    log.info("SQL данные: %s", out_path.name)
    return out_path


In [ ]:
%%writefile generated/agc_generator/cli.py
"""CLI программы 2 (generator). Открытый контур.

Пример:
    python -m agc_generator.cli --profile profile.json --scale 0.001 \
        --seed 42 --format csv --out out/

    python -m agc_generator.cli --profile profile.json --format sql --keep-gpdb
"""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

from agc_common import get_logger
from agc_generator.ddl_builder import build_ddl
from agc_generator.key_linker import generate
from agc_generator.profile_parser import load_profile
from agc_generator.writer import write_csv, write_sql

log = get_logger("generator.cli")


def build_arg_parser() -> argparse.ArgumentParser:
    p = argparse.ArgumentParser(prog="agc_generator", description="Generator (открытый контур)")
    p.add_argument("--profile", required=True, help="profile.json от profiler")
    p.add_argument("--scale", type=float, default=0.001,
                   help="scale_factor: доля от исходного числа строк (1.2M * 0.001 ~ 1200)")
    p.add_argument("--seed", type=int, default=42, help="seed для детерминизма")
    p.add_argument("--format", choices=("csv", "sql"), default="csv",
                   help="csv (файл на таблицу) или sql (батч INSERT-ов)")
    p.add_argument("--out", default="out", help="каталог/файл вывода")
    p.add_argument("--keep-gpdb", action="store_true",
                   help="сохранить GPDB-специфику (DISTRIBUTED BY, заметки о партициях)")
    return p


def main(argv: list[str] | None = None) -> int:
    args = build_arg_parser().parse_args(argv)
    profile = load_profile(args.profile)
    log.info("Профиль загружен: %d таблиц. scale=%s seed=%s format=%s",
             len(profile.tables), args.scale, args.seed, args.format)

    outdir = Path(args.out)
    outdir.mkdir(parents=True, exist_ok=True)

    # 1) DDL.
    ddl = build_ddl(profile, keep_gpdb=args.keep_gpdb)
    ddl_path = outdir / "schema.sql"
    ddl_path.write_text(ddl, encoding="utf-8")
    log.info("DDL записан: %s", ddl_path)

    # 2) Данные.
    data = generate(profile, args.scale, args.seed)
    if args.format == "csv":
        write_csv(profile, data, outdir / "data")
    else:
        write_sql(profile, data, outdir / "data.sql")

    total = sum(len(v) for v in data.values())
    log.info("Готово: %d таблиц, %d строк суммарно в %s", len(data), total, outdir)
    return 0


if __name__ == "__main__":
    sys.exit(main())


### Точки входа и вспомогательные файлы

In [ ]:
%%writefile generated/profiler.py
#!/usr/bin/env python3
"""Точка входа программы 1 (закрытый контур). См. agc_profiler/cli.py.

Запуск:  python profiler.py --tables "public.tasks,public.clients" --out profile.json
"""
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

from agc_profiler.cli import main  # noqa: E402

if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile generated/generator.py
#!/usr/bin/env python3
"""Точка входа программы 2 (открытый контур). См. agc_generator/cli.py.

Запуск:  python generator.py --profile profile.json --scale 0.001 --format csv --out out/
"""
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

from agc_generator.cli import main  # noqa: E402

if __name__ == "__main__":
    sys.exit(main())


### Документация, policy, примеры

In [ ]:
%%writefile generated/requirements.txt
# Минимальные зависимости. На закрытом контуре интернета нет — ставьте из
# заранее скачанных wheels (pip install --no-index --find-links=./wheels -r requirements.txt).

# --- Программа 1 (profiler, закрытый контур) ---
SQLAlchemy>=2.0        # адаптер БД
psycopg2-binary        # драйвер Greenplum/Postgres (или psycopg2 из системного пакета)
PyYAML>=6.0            # policy-файл

# --- Программа 2 (generator, открытый контур) ---
# Ничего обязательного сверх stdlib. Ниже — опциональные улучшения:
# Faker>=20.0          # красивее pii; без него работает встроенный фолбэк
# numpy>=1.24          # не требуется: числа генерируются через stdlib random/math


In [ ]:
%%writefile generated/README.md
# air-gap DB clone — profiler + generator

Инструмент «безопасного клонирования» БД через воздушный зазор из **двух отдельных
программ**. Через зазор едет **не выгрузка данных, а маленький человекочитаемый
`profile.json`** (схема + статистика) — его можно глазами проверить перед переносом.

```
закрытый контур (реальная БД)                 открытый контур (LLM, без реальной БД)
┌───────────────────────────┐   profile.json  ┌────────────────────────────────────┐
│  profiler                 │  ─────────────▶ │  generator                         │
│  каталог + pg_stats        │   (+ policy)    │  schema.sql + синтетические данные  │
│  → profile.json (аудит)    │                 │  (CSV или INSERT-ы)                │
└───────────────────────────┘                 └────────────────────────────────────┘
```

СУБД: **Greenplum** (MPP-форк PostgreSQL). Код совместим с разными версиями и мягко
деградирует, если GPDB-специфики (`gp_distribution_policy`, `pg_partitions`,
`pg_appendonly`) нет.

---

## Что переносить между контурами

Наружу (закрытый → открытый) едет **только `profile.json`**. Опционально — DDL от
`pg_dump --schema-only` для сверки. Реальные строки данных **не** выгружаются:
реальные значения попадают в профиль лишь для колонок, явно одобренных в policy как
`categorical_keep` (см. линтер — строгая проверка перед записью).

---

## Программа 1 — profiler (закрытый контур)

```bash
# список таблиц через запятую
python profiler.py --tables "public.tasks,public.clients" \
    --policy policy.yaml --out profile.json --sample

# либо список из CSV (колонки schema,table или schema_name,table_name)
python profiler.py --tables-csv tables.csv --policy policy.yaml --out profile.json
```

Подключение к БД (в порядке приоритета): `--dsn` → `AGC_DB_DSN`/`DB_DSN` →
`--db-config db_config.json` → подключение проекта `text2sql` (`config_db.json`/`.env`),
если инструмент запущен внутри репозитория. Движок — **read-only**
(`default_transaction_read_only=on` + `statement_timeout`).

Откуда берётся информация:
- **структура** — из системного каталога (ground truth, без скана): типы/nullability/
  default, PK/UNIQUE/NOT NULL/FK/CHECK, `gp_distribution_policy`, партиции, тип хранения;
- **статистика** — из `pg_stats` одним запросом на схему: `null_frac`, `n_distinct`
  (знак сохраняем!), `avg_width`, `most_common_vals`/`freqs`, `histogram_bounds`;
- **число строк** — из `reltuples` (помечается как оценка);
- **сэмпл** (`--sample`) — только для инспекции форматов коротких текстовых полей;
  для кардинальности **не** используется (см. оговорку о смещении в `sampler.py`);
- `--recompute-missing` — точечно досчитать `null_frac`/`n_distinct` агрегатами там,
  где `pg_stats` пуст (это **скан**, по умолчанию выключено, логируется предупреждением).

### Формат policy-YAML (whitelist)

По умолчанию **каждая** колонка чувствительна и синтезируется. `categorical_keep`
недостижим автоматически — только явно в policy (граница безопасности).

```yaml
version: 1
columns:
  "public.tasks.task_type": categorical_keep                 # хранит реальные значения
  "public.clients.name":    {class: pii, generator: full_name}
  "public.tasks.amount":    {class: sensitive_numeric, dist: lognormal, avg_hint: "~5e4"}
tables:
  "public.tasks":
    order_groups: [["created_at", "updated_at"]]             # created <= updated
```

Классы: `categorical_keep` | `sensitive_numeric` | `pii` | `key` | `datetime` | `sensitive`.

---

## Программа 2 — generator (открытый контур)

```bash
python generator.py --profile profile.json --scale 0.001 --seed 42 \
    --format csv --out out/                # CSV на таблицу  (+ out/schema.sql)

python generator.py --profile profile.json --scale 0.001 \
    --format sql --keep-gpdb               # батч INSERT-ов + DISTRIBUTED BY
```

- `--scale` — доля от исходного числа строк (1.2M × 0.001 ≈ 1200);
- `--keep-gpdb` — сохранить `DISTRIBUTED BY` и заметки о партициях (иначе всё heap);
- согласованное масштабирование кардинальности и FK: **все джойны резолвятся**
  (FK берутся из уже сгенерированных родительских ключей);
- без ML-синтезаторов (SDV/CTGAN) — лёгкий детерминированный генератор по профилю.

---

## Установка (закрытый контур без интернета)

```bash
pip install --no-index --find-links=./wheels -r requirements.txt
```

Обязательны только `SQLAlchemy`, `psycopg2`, `PyYAML` (для profiler). Generator
работает на stdlib; `Faker`/`numpy` — опциональные улучшения.

---

## Структура

```
agc_common.py            общие утилиты (валидация идентификаторов, логирование)
agc_profiler/            ПРОГРАММА 1
  db.py                  подключение (read-only), переиспользует коннект проекта
  catalog_reader.py      структура из системного каталога
  stats_reader.py        pg_stats (+ парсер PG-массивов, досчёт по агрегатам)
  sampler.py             адаптивный сэмпл форматов (с оговоркой о смещении)
  classifier.py          авто-эвристики (только ПРЕДЛАГАЮТ класс)
  policy.py              whitelist чувствительности (финальное решение)
  linter.py              строгая проверка утечек реальных значений
  profile_builder.py     сборка profile.json
  cli.py                 CLI
agc_generator/           ПРОГРАММА 2
  profile_parser.py      парсер profile.json
  ddl_builder.py         schema.sql (топосортировка по FK)
  value_generators.py    генераторы значений по классам
  key_linker.py          суррогатные PK, FK, масштабирование кардинальности
  writer.py              CSV / INSERT-ы
  cli.py                 CLI
profiler.py, generator.py   точки входа
policy.example.yaml, tables.example.csv, examples/profile.example.json
```


In [ ]:
%%writefile generated/policy.example.yaml
# Policy — whitelist чувствительности (граница безопасности).
#
# По умолчанию КАЖДАЯ колонка чувствительна и синтезируется. Реальные значения
# (most_common_vals/histogram_bounds) уходят в профиль ТОЛЬКО для колонок,
# явно помеченных здесь как categorical_keep. Забыли пометить новую колонку с ФИО —
# она НЕ утечёт (по умолчанию синтезируется), в отличие от blacklist-подхода.
#
# Классы: categorical_keep | sensitive_numeric | pii | key | datetime | sensitive
version: 1

columns:
  # Бизнес-категории — сохраняем реальные distinct-значения и частоты (для GROUP BY).
  "public.tasks.task_type": categorical_keep
  "public.tasks.status":    categorical_keep
  "public.clients.region":  categorical_keep

  # Явно чувствительные — синтез (это и так дефолт, но фиксируем генератор).
  "public.clients.client_name": {class: pii, generator: full_name}
  "public.clients.email":       {class: pii, generator: email}
  "public.clients.phone":       {class: pii, generator: phone}

  # Числовые суммы — форма распределения без реальных значений.
  "public.tasks.amount":     {class: sensitive_numeric, dist: lognormal, avg_hint: "~5e4"}
  "public.payments.amount":  {class: sensitive_numeric, dist: lognormal, avg_hint: "~1e4"}

tables:
  # Порядок связанных дат: created_at <= updated_at.
  "public.tasks":
    order_groups: [["created_at", "updated_at"]]


In [ ]:
%%writefile generated/tables.example.csv
schema,table
public,clients
public,tasks
public,payments


In [ ]:
%%writefile generated/examples/profile.example.json
{
  "profile_version": 1,
  "generated_at": "2026-07-20T00:00:00+00:00",
  "source_dialect": "greenplum",
  "note": "Пример профиля: реальные значения только в categorical_keep.",
  "tables": [
    {
      "schema": "public",
      "table": "clients",
      "relkind": "table",
      "is_view": false,
      "storage": "heap",
      "distributed_by": ["id"],
      "partitioned_by": [],
      "row_count": {"value": 500000, "estimated": true},
      "constraints": {
        "pk": ["id"],
        "uniques": [["email"]],
        "fks": [],
        "checks": []
      },
      "defaults": {},
      "not_null": ["id", "client_name"],
      "columns": {
        "id":          {"pg_type": "bigint", "policy": "key", "role": "pk", "n_distinct": -1.0, "null_frac": 0.0},
        "client_name": {"pg_type": "text", "policy": "pii", "null_frac": 0.0, "avg_width": 18, "generator": "full_name", "unique_like": false},
        "email":       {"pg_type": "character varying(120)", "policy": "pii", "null_frac": 0.02, "avg_width": 22, "generator": "email", "unique_like": true, "char_len": 120},
        "phone":       {"pg_type": "character varying(20)", "policy": "pii", "null_frac": 0.1, "avg_width": 12, "generator": "phone", "unique_like": false, "char_len": 20},
        "region":      {"pg_type": "text", "policy": "categorical_keep", "null_frac": 0.01, "n_distinct": 4,
                        "values": [["Центр", 0.4], ["Юг", 0.25], ["Сибирь", 0.2], ["Урал", 0.14]]}
      }
    },
    {
      "schema": "public",
      "table": "tasks",
      "relkind": "table",
      "is_view": false,
      "storage": "append_optimized_column",
      "distributed_by": ["task_id"],
      "partitioned_by": ["created_at"],
      "row_count": {"value": 1200000, "estimated": true},
      "constraints": {
        "pk": ["task_id"],
        "uniques": [],
        "fks": [{"columns": ["client_id"], "ref_schema": "public", "ref_table": "clients", "ref_columns": ["id"]}],
        "checks": ["CHECK ((amount >= (0)::numeric))"]
      },
      "defaults": {},
      "not_null": ["task_id", "client_id", "task_type"],
      "columns": {
        "task_id":    {"pg_type": "bigint", "policy": "key", "role": "pk", "n_distinct": -1.0, "null_frac": 0.0},
        "client_id":  {"pg_type": "bigint", "policy": "key", "role": "fk", "n_distinct": -0.3, "null_frac": 0.0,
                       "references": {"schema": "public", "table": "clients", "column": "id"},
                       "fanout": {"dist": "uniform"}},
        "task_type":  {"pg_type": "text", "policy": "categorical_keep", "null_frac": 0.0, "n_distinct": 4,
                       "values": [["отток", 0.4], ["привлечение", 0.3], ["сопровождение", 0.2], ["прочее", 0.1]]},
        "status":     {"pg_type": "text", "policy": "categorical_synth", "null_frac": 0.0, "n_distinct": 5,
                       "freqs": [0.5, 0.2, 0.15, 0.1, 0.05]},
        "amount":     {"pg_type": "numeric(14,2)", "policy": "sensitive_numeric", "null_frac": 0.05, "n_distinct": -0.8,
                       "precision": 14, "scale": 2, "dist": "lognormal", "avg_hint": "~5e4"},
        "created_at": {"pg_type": "timestamp without time zone", "policy": "datetime", "null_frac": 0.0,
                       "range": {"start": "2020-01-01", "end": "2024-12-31"}, "order_group": true},
        "updated_at": {"pg_type": "timestamp without time zone", "policy": "datetime", "null_frac": 0.1,
                       "range": {"start": "2020-01-01", "end": "2024-12-31"}, "order_group": true}
      }
    },
    {
      "schema": "public",
      "table": "payments",
      "relkind": "table",
      "is_view": false,
      "storage": "heap",
      "distributed_by": ["payment_id"],
      "partitioned_by": [],
      "row_count": {"value": 900000, "estimated": true},
      "constraints": {
        "pk": ["payment_id"],
        "uniques": [],
        "fks": [{"columns": ["task_id"], "ref_schema": "public", "ref_table": "tasks", "ref_columns": ["task_id"]}],
        "checks": []
      },
      "defaults": {},
      "not_null": ["payment_id", "task_id"],
      "columns": {
        "payment_id": {"pg_type": "bigint", "policy": "key", "role": "pk", "n_distinct": -1.0, "null_frac": 0.0},
        "task_id":    {"pg_type": "bigint", "policy": "key", "role": "fk", "n_distinct": -0.6, "null_frac": 0.0,
                       "references": {"schema": "public", "table": "tasks", "column": "task_id"},
                       "fanout": {"dist": "uniform"}},
        "amount":     {"pg_type": "numeric(12,2)", "policy": "sensitive_numeric", "null_frac": 0.0, "n_distinct": -0.9,
                       "precision": 12, "scale": 2, "dist": "lognormal", "avg_hint": "~1e4"},
        "paid_at":    {"pg_type": "date", "policy": "datetime", "null_frac": 0.0,
                       "range": {"start": "2020-01-01", "end": "2024-12-31"}}
      }
    }
  ]
}


## 3. profiler — закрытый контур

Переиспользуем подключение проекта `text2sql` (`config_db.json`/`.env`). Если живой БД нет — берём готовый пример профиля, чтобы продемонстрировать generator. Движок read-only; реальные строки наружу не выгружаются.

In [ ]:
import sys, json
sys.path.insert(0, str(GEN))                         # agc_* модули
proj_src = NB_DIR.parent / "src"                     # переиспользуем коннект проекта
if proj_src.exists():
    sys.path.insert(0, str(proj_src))

from agc_profiler.cli import parse_tables_arg, parse_tables_csv
from agc_profiler.policy import Policy
from agc_profiler.profile_builder import build_profile
from agc_profiler.linter import check_profile
from agc_profiler.db import make_engine

tables = parse_tables_csv(TABLES_CSV) if TABLES_CSV else parse_tables_arg(TABLES)
policy = Policy.load(POLICY_YAML or None)
profile_path = OUT / "profile.json"

try:
    engine = make_engine()                           # DSN из проекта / env / db_config.json
    engine.connect().close()
    profile = build_profile(engine, tables, policy, sample=SAMPLE)
    check_profile(profile)                           # строгий линтер утечек
    profile_path.write_text(json.dumps(profile, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"profiler отработал вживую → {profile_path}")
except Exception as e:
    print(f"[i] Живое подключение к БД недоступно ({type(e).__name__}: {e}).")
    print("    Использую готовый пример профиля для демонстрации генератора.")
    import shutil
    shutil.copy(GEN / "examples" / "profile.example.json", profile_path)
    print(f"    Профиль: {profile_path}")

## 4. Линтер профиля — аудит утечек

Главная точка аудита: если колонка **не** `categorical_keep`, но в её профиле есть реальные значения — падаем. Здесь же — сводка классов по таблицам.

In [ ]:
prof = json.loads(profile_path.read_text(encoding="utf-8"))
check_profile(prof)
print("Линтер OK: реальные значения только в categorical_keep.\n")
for t in prof["tables"]:
    kinds = {}
    for c in t["columns"].values():
        kinds[c["policy"]] = kinds.get(c["policy"], 0) + 1
    rc = t["row_count"]["value"]
    print(f"  {t['schema']}.{t['table']}: rows~{rc}  classes={kinds}")

## 5. generator — открытый контур

`schema.sql` (DDL) + синтетические данные. FK берутся из уже сгенерированных родительских ключей — все джойны резолвятся; кардинальность масштабируется согласованно.

In [ ]:
from agc_generator.profile_parser import load_profile
from agc_generator.ddl_builder import build_ddl
from agc_generator.key_linker import generate
from agc_generator.writer import write_csv, write_sql

gprof = load_profile(profile_path)
(OUT / "schema.sql").write_text(build_ddl(gprof, keep_gpdb=KEEP_GPDB), encoding="utf-8")
data = generate(gprof, SCALE_FACTOR, SEED)
if OUTPUT_FORMAT == "csv":
    write_csv(gprof, data, OUT / "data")
else:
    write_sql(gprof, data, OUT / "data.sql")

total = sum(len(v) for v in data.values())
print(f"\nГотово: {len(data)} таблиц, {total} строк суммарно → {OUT}")

## 6. Предпросмотр результата

In [ ]:
print("=== generated/ (инструмент) ===")
for p in sorted(GEN.rglob("*")):
    if p.is_file() and "output" not in p.parts:
        print("  ", p.relative_to(NB_DIR))
print("\n=== output/ (артефакты запуска) ===")
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print("  ", p.relative_to(NB_DIR), f"{p.stat().st_size} B")
print("\n--- schema.sql (первые строки) ---")
print("\n".join((OUT / "schema.sql").read_text(encoding="utf-8").splitlines()[:22]))

## Запуск как отдельных CLI (для реального переноса между контурами)

```bash
# закрытый контур:
python generated/profiler.py --tables "public.tasks,public.clients" \
    --policy generated/policy.example.yaml --out profile.json --sample

# перенести profile.json через зазор, затем на открытом контуре:
python generated/generator.py --profile profile.json --scale 0.001 \
    --seed 42 --format csv --out out/
```

Подробности — в `generated/README.md`.